# Retake Coach: run the demo page

**Cost: $0 for checking photos** (DINOv2, and DINOv3 for where an obstruction is, run here; each instruction is a
standard sentence built from what they measured). Muse Spark writes the coach's note at the end of a visit or batch,
from counts in words, never photos. It is capped in the last cell: $0.25 a day, 60 requests a day and $1.00 in total for
this runtime. Set `USE_MUSE = False` to spend nothing; the note is then a standard summary.
**Runtime:** GPU is fastest (T4 is enough). CPU works too, at about 1–2 seconds a photo.

Needs `quality/coach_model.npz`, `quality/build_features.npz` and (optional) `quality/locator.npz` and `quality/obstruction_map.npz` in your Drive folder, plus `MODEL_API_KEY` (Muse Spark) and `HF_TOKEN` (DINOv3, for the obstruction map) in Colab Secrets.
The last cell prints two links: the one-photo page (with the visit checklist) and the many-photos page, which also shows photos that land in `eyer_inbox/` in your Drive.

In [ ]:
import os, sys, glob, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'
MODEL = f'{D}/quality/coach_model.npz'
LOCATOR = f'{D}/quality/locator.npz'
MAP = f'{D}/quality/obstruction_map.npz'     # made by obstruction_map_build.ipynb
assert os.path.exists(MODEL), 'coach_model.npz is missing from your Drive quality folder'
APP = '/content/retake_coach'
os.makedirs(f'{APP}/static', exist_ok=True)
subprocess.run('pip install -q fastapi uvicorn python-multipart openai', shell=True, check=True)

In [ ]:
%%writefile /content/retake_coach/coach_core.py
"""Retake coach: is this fundus photo good enough to grade, and if not, what should the photographer fix?

It judges photo quality only and makes no diagnosis.

- Detection: Meta's DINOv2 (facebook/dinov2-base, frozen) runs locally. Four small logistic heads read its
  features plus three simple image checks: not good enough, focus, aim, obstruction. The heads are trained on
  DeepDRiD's photo-quality labels (CC BY-SA 4.0, https://github.com/deepdrdoc/DeepDRiD).
- Guard: an energy on the same features (a Gaussian model of ordinary fundus photos, i.e. Mahalanobis distance)
  flags inputs that do not look like a regular fundus photo, such as a widefield image or a random picture.
- Where: the disc and macula locator (DINOv2 patch tokens) gives aim directions; the obstruction map (the
  obstruction head's own per-patch split, on DINOv3 patch tokens) says where a shadow or glare is.
- Instructions: standard sentences built from those measured facts, on this computer.
- Coach's note: Muse Spark (Meta Model API) writes a short note at the end of a session from the session's counts in
  words; no photo is sent. A hard cap on requests and dollars applies; past the cap, or on any error, a fixed summary
  is used. (Wording: Muse Spark can also rewrite a single instruction; the app does not use that, see the README.)
"""
import base64
import io
import json
import re
from collections import Counter

import numpy as np
from PIL import Image
from scipy import ndimage

S = 448
PROBLEMS = ("focus", "aim", "obstruction")
DESCRIBE = {
    "focus": "out of focus or hazy: the small vessels are not sharp",
    "aim": "badly aimed: the optic disc and the macula are not both near the centre of the field",
    "obstruction": "partly covered: eyelid or eyelash shadow, glare, reflections or dust over the retina",
}
TEMPLATES = {
    "focus": "Refocus until the small vessels look sharp, then retake.",
    "aim": "Re-aim so the view is centred on the macula or on the optic disc, then retake.",
    "obstruction": "Ask the patient to open wide (lift the eyelid if needed), wipe the lens, then retake.",
}
GENERIC = "Not good enough to grade: check focus, aim and eyelids, then retake."
MINOR = {"focus": "Focus could be sharper", "aim": "The aim is a little off centre", "obstruction": "Part of the view is shaded"}


def keep_text(minor, aim_text=None):
    if not minor:
        return "Good enough to grade. Move on to the next photo."
    where = aim_text.split(". ")[0] if (minor[0] == "aim" and aim_text and not aim_text.startswith(("Centre", "Re-aim"))) else MINOR[minor[0]]
    return f"Good enough to grade. {where}; this photo is still usable."
NOT_FUNDUS = "This does not look like a regular fundus photo. Check the camera mode or the file."
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])


# ---------- image preparation (identical to the pipeline the heads were trained on) ----------

def _open(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if isinstance(src, np.ndarray):
        return Image.fromarray(src).convert("RGB")
    return Image.open(src).convert("RGB")


def load_square(src, geom=False):
    """Crop to the retina, pad to a square, resize to 448 px. Returns float32 in [0, 1]. src: path, PIL image or uint8 array.
    geom=True also returns how original pixel coordinates map into the square (see to_square)."""
    img = _open(src)
    W0, H0 = img.size
    img.thumbnail((1024, 1024))
    a = np.asarray(img)
    ys, xs = np.nonzero(a.max(-1) > 15)
    y0 = x0 = 0
    if len(ys):                                   # an all-black frame keeps its full size
        y0, x0 = ys.min(), xs.min()
        a = a[y0:ys.max() + 1, x0:xs.max() + 1]
    h, w = a.shape[:2]
    side = max(h, w)
    sq = np.zeros((side, side, 3), np.uint8)
    sq[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = a
    x = np.asarray(Image.fromarray(sq).resize((S, S), Image.BILINEAR)).astype(np.float32) / 255
    if not geom:
        return x
    return x, dict(sx=img.size[0] / W0, sy=img.size[1] / H0, x0=int(x0), y0=int(y0),
                   px=(side - w) // 2, py=(side - h) // 2, side=side)


def to_square(X, Y, g):
    """Original pixel (X, Y) -> position in the square photo, as fractions (0..1) of its width and height."""
    return (X * g["sx"] - g["x0"] + g["px"]) / g["side"], (Y * g["sy"] - g["y0"] + g["py"]) / g["side"]


def retina_mask(x):
    return x.max(-1) > 15 / 255


def simple_checks(x):
    """Sharpness (Laplacian variance), brightness and contrast of the green channel inside the retina."""
    g, m = x[..., 1], ndimage.binary_erosion(retina_mask(x), iterations=8)
    if m.sum() < 100:                             # nearly black frame: measure the whole picture
        m = np.ones_like(m)
    return [ndimage.laplace(g)[m].var(), g[m].mean(), g[m].std()]


class Encoder:
    """Frozen DINOv2 or DINOv3: class token + mean patch token (2 x 768 numbers per photo for the base models).
    Register tokens (DINOv3 has 4) sit between the class token and the patches and are skipped."""

    def __init__(self, name="facebook/dinov2-base", device=None, dtype=None):
        import torch
        from transformers import AutoModel
        self.torch, self.name = torch, name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = dtype or (torch.float16 if self.device == "cuda" else torch.float32)
        self.model = AutoModel.from_pretrained(name).to(self.device, self.dtype).eval()
        ps = self.model.config.patch_size
        self.patch = ps[0] if isinstance(ps, (list, tuple)) else ps

    def __call__(self, xs, bs=32, patches=False):
        """patches=True also returns the patch tokens as a (n, g, g, 768) float16 grid (g = 448 / patch size)."""
        out, grids = [], []
        g = S // self.patch
        with self.torch.no_grad():
            for i in range(0, len(xs), bs):
                b = self.torch.tensor((np.stack(xs[i:i + bs]) - MEAN) / STD).permute(0, 3, 1, 2)
                h = self.model(pixel_values=b.to(self.device, self.dtype)).last_hidden_state
                p = h[:, h.shape[1] - g * g:]                       # patch tokens are the last g*g
                out.append(self.torch.cat([h[:, 0], p.mean(1)], 1).float().cpu())
                if patches:
                    grids.append(p.reshape(len(b), g, g, -1).half().cpu())
        X = self.torch.cat(out).numpy()
        return (X, self.torch.cat(grids).numpy()) if patches else X


def describe(srcs, encoder, bs=32, workers=8):
    """Photos -> (DINOv2 features, simple checks). Loads in parallel, encodes in batches."""
    from concurrent.futures import ThreadPoolExecutor
    X, C = [], []
    for i in range(0, len(srcs), bs):
        with ThreadPoolExecutor(workers) as ex:
            xs = list(ex.map(load_square, srcs[i:i + bs]))
        X.append(encoder(xs, bs))
        C += [simple_checks(x) for x in xs]
    return np.concatenate(X), np.array(C)


# ---------- detector heads and energy guard ----------

def gaussian(X):
    """Mean and whitening matrix of a shrunk (Ledoit-Wolf) Gaussian over feature rows."""
    from sklearn.covariance import LedoitWolf
    lw = LedoitWolf().fit(np.asarray(X, np.float64))
    lam, V = np.linalg.eigh(lw.covariance_)                    # whiten from the covariance itself:
    lam = np.maximum(lam, lam.max() * 1e-6)                    # tiny directions are floored, never inverted to inf
    return lw.location_, V / np.sqrt(lam)


def energy(X, mean, white):
    return (((X - mean) @ white) ** 2).sum(1) / X.shape[1]


def camera_component(X_cam, name, factor=1.1):
    """A Gaussian over one camera's own photos. Its limit is the largest leave-one-out energy among them, times factor,
    so each photo is judged by a model that never saw it."""
    X_cam = np.asarray(X_cam, np.float64)
    assert len(X_cam) >= 5, "add at least 5 ordinary photos from the camera"
    loo = []
    for k in range(len(X_cam)):
        m, w = gaussian(np.delete(X_cam, k, 0))
        loo.append(energy(X_cam[k:k + 1], m, w)[0])
    m, w = gaussian(X_cam)
    return dict(name=name, mean=m, white=w, thr=float(max(loo) * factor))


def _sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


class Coach:
    HEADS = ("retake",) + PROBLEMS
    RULES = ("overall", "problems", "both")   # what triggers a retake: the overall head, any problem head, or both

    def __init__(self, mu, sd, W, b, thr, g_mean=None, g_white=None, g_thr=None, rule="overall", locator=None,
                 backbone="facebook/dinov2-base"):
        assert rule in self.RULES
        self.mu, self.sd, self.W, self.b, self.thr, self.rule = mu, sd, W, b, np.asarray(thr, float), rule
        self.backbone = backbone                 # the encoder whose features these heads read
        self.g_mean, self.g_white, self.g_thr = g_mean, g_white, g_thr
        self.cameras = []                         # onboarded cameras: dicts of name, mean, white, thr
        self.locator = locator                   # optional Locator; saved in its own file (use attach_locator)

    def attach_locator(self, locator):
        """Use the locator only if it was fitted on the same encoder's patch tokens."""
        if locator.backbone != self.backbone:
            print(f"locator reads {locator.backbone}, heads read {self.backbone}: aim directions off")
            return False
        self.locator = locator
        return True

    @classmethod
    def fit(cls, X, checks, labels, C=0.05):
        """labels: dict head -> bool array; C: one value or a dict head -> value (smaller = stronger regularization).
        Thresholds start at 0.5; set coach.thr from validation photos."""
        from sklearn.linear_model import LogisticRegression
        mu, sd = checks.mean(0), checks.std(0) + 1e-12
        F = np.c_[X, (checks - mu) / sd]
        W, b = [], []
        for h in cls.HEADS:
            clf = LogisticRegression(C=C[h] if isinstance(C, dict) else C, max_iter=5000).fit(F, labels[h])
            W.append(clf.coef_[0])
            b.append(clf.intercept_[0])
        return cls(mu, sd, np.array(W), np.array(b), np.full(len(cls.HEADS), 0.5))

    def probs(self, X, checks):
        """(n, 4) probabilities in HEADS order."""
        return _sigmoid(np.c_[X, (checks - self.mu) / self.sd] @ self.W.T + self.b)

    def set_guard(self, X_ref, X_cal, quantile=0.99):
        """Energy guard fitted on ordinary fundus photos (X_ref); threshold = the 99th percentile on other ones (X_cal)."""
        self.g_mean, self.g_white = gaussian(X_ref)
        self.g_thr = float(np.quantile(self.energy(X_cal), quantile))
        return self

    def energy(self, X):
        """Mahalanobis energy per feature dimension: about 1 for typical fundus photos, large for unfamiliar inputs."""
        return energy(X, self.g_mean, self.g_white)

    def add_camera(self, X_cam, name, factor=1.1):
        """Onboard a new camera from its own ordinary photos, no labels: a second Gaussian over just those photos.
        A photo passes the guard if either the general model or a camera's model finds it familiar."""
        self.cameras.append(camera_component(X_cam, name, factor))
        return self

    def familiar(self, X):
        """(n,) bool: does each photo look like a regular fundus photo to the general model or to an onboarded camera?"""
        if self.g_mean is None:
            return np.ones(len(X), bool)
        ok = self.energy(X) <= self.g_thr
        for c in self.cameras:
            ok |= energy(X, c["mean"], c["white"]) <= c["thr"]
        return ok

    def verdict(self, X, checks, patches=None, max_problems=2):
        """patches (from Encoder(..., patches=True)) adds measured disc and macula positions when a locator is set."""
        P = self.probs(X, checks)
        E = self.energy(X) if self.g_mean is not None else np.zeros(len(X))
        OK = self.familiar(X)
        L = self.locator.locate(patches) if (self.locator is not None and patches is not None) else [None] * len(X)
        out = []
        for p, e, loc, ok in zip(P, E, L, OK):
            fundus = bool(ok)
            margin = _logit(p[1:]) - _logit(self.thr[1:])          # how far each problem head is past its threshold
            overall, any_problem = bool(p[0] >= self.thr[0]), bool((margin > 0).any())
            retake = {"overall": overall, "problems": any_problem, "both": overall and any_problem}[self.rule]
            order = [int(i) for i in np.argsort(-margin)]
            problems = [PROBLEMS[i] for i in order if margin[i] > 0][:max_problems] if retake else []
            if retake and not problems:                              # always give one concrete thing to fix
                problems = [PROBLEMS[order[0]]]
            v = dict(looks_like_fundus=fundus, retake=retake and fundus, problems=problems if fundus else [], minor=[],
                     p_not_good=float(p[0]), scores={k: float(v) for k, v in zip(PROBLEMS, p[1:])}, energy=float(e))
            if loc is not None and fundus:
                dd = self.locator.dd
                # a kept photo gets an aim note only when the aim head and the measured positions both say off centre
                both_found = loc["disc"] is not None and loc["macula"] is not None
                if not retake and margin[1] > 0 and both_found and aim_advice(loc, dd) is not None:
                    v["minor"] = ["aim"]
                v.update(disc=loc["disc"], macula=loc["macula"], dd=dd, positions=positions_text(loc, dd),
                         aim_text=aim_instruction(loc, dd) if "aim" in v["problems"] + v["minor"] else None,
                         **eye_and_field(loc, dd))
            out.append(v)
        return out

    def save(self, path):
        g = {} if self.g_mean is None else dict(g_mean=self.g_mean, g_white=self.g_white.astype(np.float32), g_thr=self.g_thr)
        for i, c in enumerate(self.cameras):
            g.update({f"cam{i}_name": np.array(c["name"]), f"cam{i}_mean": c["mean"], f"cam{i}_white": c["white"].astype(np.float32),
                      f"cam{i}_thr": c["thr"]})
        np.savez(path, mu=self.mu, sd=self.sd, W=self.W, b=self.b, thr=self.thr, heads=np.array(self.HEADS),
                 rule=np.array(self.rule), backbone=np.array(self.backbone), **g)

    @classmethod
    def load(cls, path):
        z = np.load(path)
        assert tuple(z["heads"]) == cls.HEADS
        g = dict(g_mean=z["g_mean"], g_white=z["g_white"].astype(np.float64), g_thr=float(z["g_thr"])) if "g_mean" in z else {}
        c = cls(z["mu"], z["sd"], z["W"], z["b"], z["thr"], rule=str(z["rule"]) if "rule" in z else "overall",
                backbone=str(z["backbone"]) if "backbone" in z else "facebook/dinov2-base", **g)
        i = 0
        while f"cam{i}_mean" in z:
            c.cameras.append(dict(name=str(z[f"cam{i}_name"]), mean=z[f"cam{i}_mean"], white=z[f"cam{i}_white"].astype(np.float64),
                                  thr=float(z[f"cam{i}_thr"])))
            i += 1
        return c


# ---------- where are the optic disc and the macula? ----------

class Locator:
    """A linear map from DINOv2 patch tokens to two heatmaps (optic disc, fovea), fitted by ridge regression on
    IDRiD's expert-marked centres. Positions are fractions (0..1) of the square photo; dd is one disc diameter
    in the same units, so distances can be read the way DeepDRiD's field-definition score reads them."""
    NAMES = ("disc", "macula")

    def __init__(self, W, peak_min, dd, backbone="facebook/dinov2-base"):
        self.W, self.peak_min, self.dd = np.asarray(W, np.float32), np.asarray(peak_min, float), float(dd)
        self.backbone = backbone

    @staticmethod
    def design(T):
        """(n, g, g, d) tokens -> (n, g*g, d+1) with a constant column for the bias."""
        T = np.asarray(T, np.float32).reshape(T.shape[0], -1, T.shape[-1])
        return np.concatenate([T, np.ones(T.shape[:2] + (1,), np.float32)], -1)

    def heatmaps(self, T, bs=32):
        g = T.shape[1]
        return np.concatenate([(self.design(T[i:i + bs]) @ self.W).reshape(-1, g, g, len(self.NAMES))
                               for i in range(0, len(T), bs)])

    def locate(self, T):
        out = []
        for hm in self.heatmaps(T):
            g, found = hm.shape[0], {}
            for k, name in enumerate(self.NAMES):
                m = ndimage.uniform_filter(hm[..., k], 3)
                i, j = np.unravel_index(np.argmax(m), m.shape)
                i0, i1, j0, j1 = max(i - 1, 0), min(i + 2, g), max(j - 1, 0), min(j + 2, g)
                win = np.clip(hm[i0:i1, j0:j1, k], 0, None) + 1e-9          # sub-patch refinement
                ii, jj = np.mgrid[i0:i1, j0:j1]
                u, v = ((jj * win).sum() / win.sum() + 0.5) / g, ((ii * win).sum() / win.sum() + 0.5) / g
                found[name] = (float(u), float(v)) if m[i, j] >= self.peak_min[k] else None
                found[name + "_peak"] = float(m[i, j])
            out.append(found)
        return out

    def save(self, path):
        np.savez(path, W=self.W, peak_min=self.peak_min, dd=self.dd, backbone=np.array(self.backbone))

    @classmethod
    def load(cls, path):
        z = np.load(path)
        return cls(z["W"], z["peak_min"], float(z["dd"]), str(z["backbone"]) if "backbone" in z else "facebook/dinov2-base")



# ---------- where is the obstruction? the obstruction head's own patch map ----------

DINOV3 = "facebook/dinov3-vitb16-pretrain-lvd1689m"


def patch_mask(x, g):
    """Which of the g x g patches are mostly retina, not the black surround."""
    k = S // g
    return retina_mask(x).reshape(g, k, g, k).mean((1, 3)) > 0.5


def smooth_on(r, m, sigma=1.0):
    """Average each patch with its retina neighbours only, so the black rim does not bleed in."""
    num = ndimage.gaussian_filter(np.where(m, r, 0.0), sigma)
    den = ndimage.gaussian_filter(m.astype(float), sigma)
    return np.where(m, num / np.maximum(den, 1e-6), 0.0)


def _overlay_png(area, light=False):
    """The marked area as a see-through blue layer with a white edge, the size of the square photo. A large area gets a
    lighter fill so the photo underneath stays visible."""
    edge = area & ~ndimage.binary_erosion(area, iterations=2)
    rgba = np.zeros(area.shape + (4,), np.uint8)
    rgba[area] = (40, 140, 255, 45 if light else 105)
    rgba[edge] = (255, 255, 255, 230)
    buf = io.BytesIO()
    Image.fromarray(rgba, "RGBA").save(buf, "PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


class ObstructionMap:
    """Where the obstruction is, read from the obstruction head itself.
    Part of the head's score is its weight on the mean patch token, so that part splits exactly into one number per
    patch. Each patch is compared with the same place on a typical clean photo (DeepDRiD artifact score 0), and patches
    above what 1% of clean patches reach are marked. On DeepDRiD validation photos with DINOv3, the marked share follows
    the artifact score (Spearman 0.81), and the strongest spot lands on a pasted shadow 85% and a pasted glare 80% of
    the time (chance 7%); results/obstruction_map_report.txt."""

    HINT_SHARE = 0.054   # measured: 95% of artifact-0 DeepDRiD validation photos mark less (obstruction_map_build_report.txt)

    def __init__(self, w, base, thr, backbone=DINOV3, min_share=0.0, hint_share=None):
        self.w, self.base, self.thr = np.asarray(w, np.float32), np.asarray(base, np.float32), float(thr)
        self.backbone, self.min_share = backbone, float(min_share)   # smaller marked areas are what clean photos show
        # on a retake for another reason, a marked area larger than this is shown too: clear photos rarely mark this much
        self.hint_share = float(self.HINT_SHARE if hint_share is None else hint_share)

    @staticmethod
    def head_weights(coach):
        """The obstruction head's weights on the mean patch token (features are [class token, mean patch, 3 checks])."""
        n = (coach.W.shape[1] - 3) // 2
        return np.asarray(coach.W[Coach.HEADS.index("obstruction"), n:2 * n], np.float32)

    def raw(self, grids):
        """(..., g, g, d) patch tokens -> (..., g, g) share of the obstruction score from each patch."""
        return np.asarray(grids, np.float32) @ self.w

    @classmethod
    def fit(cls, w, maps, masks, backbone, quantile=99.0, share_quantile=None):
        """maps: raw maps of clean photos (n, g, g); masks: their retina patches.
        Sets the typical map and the level a patch must pass (what 1% of clean patches reach). share_quantile, if given,
        also sets the smallest marked area worth reporting (e.g. 95: what 95% of clean photos stay under). The app runs
        the map only after the obstruction check has asked for a retake, so by default there is no such floor."""
        masks = np.asarray(masks)
        base = np.median(maps, 0)
        smoothed = np.array([smooth_on(m - base, pm) for m, pm in zip(maps, masks)])
        thr = max(float(np.percentile(smoothed[masks], quantile)), 1e-6)
        shares = [((sm > thr) & pm).sum() / max(1, pm.sum()) for sm, pm in zip(smoothed, masks)]
        return cls(w, base, thr, backbone, float(np.percentile(shares, share_quantile)) if share_quantile else 0.0,
                   hint_share=float(np.percentile(shares, 95)))

    def strength(self, x, raw):
        """Each patch's push toward 'something is in the way', relative to a clean photo, and the retina patches."""
        pm = patch_mask(x, raw.shape[0])
        return smooth_on(raw - self.base, pm), pm

    def locate(self, x, grid, exclude=None):
        """x: the square photo; grid: its (g, g, d) patch tokens from the map's backbone.
        Returns where (side of the picture, 'centre' or 'most'), tone (darker, brighter or None), share and an overlay.
        exclude: (u, v, r) leaves out patches within r of (u, v), e.g. around the macula, which is naturally darker."""
        return self.locate_raw(x, self.raw(grid), exclude)

    def locate_raw(self, x, raw, exclude=None):
        """The same from an already computed raw map (g, g)."""
        s, pm = self.strength(x, raw)
        g = s.shape[0]
        marked = (s > self.thr) & pm
        if exclude:
            c = (np.arange(g) + 0.5) / g
            marked &= np.hypot(c[None, :] - exclude[0], c[:, None] - exclude[1]) > exclude[2]
        share = float(marked.sum() / max(1, pm.sum()))
        out = dict(where=None, tone=None, share=share, overlay=None)
        if not marked.any() or share <= self.min_share:
            return out
        lab, _ = ndimage.label(marked)                  # the marked area that holds the strongest spot
        top = np.unravel_index(np.argmax(np.where(marked, s, -np.inf)), s.shape)
        cy, cx = ndimage.center_of_mass(lab == lab[top])
        u, v = (cx + 0.5) / g, (cy + 0.5) / g
        out["where"] = "most" if share >= 0.5 else "centre" if np.hypot(u - 0.5, v - 0.5) < 0.15 else side_of(u, v)
        ret = retina_mask(x)
        up = np.array(Image.fromarray(s.astype(np.float32)).resize((S, S), Image.BILINEAR))
        k = S / g                                          # the marked patches at photo size, with rounded edges
        area = ndimage.gaussian_filter(np.array(Image.fromarray(marked.astype(np.uint8) * 255).resize((S, S), Image.NEAREST)) / 255.0,
                                       k / 2) > 0.5
        area &= ret
        rest, lum = ret & ~area, x.mean(-1)
        if area.any() and rest.sum() > 0.1 * ret.sum():          # tone of the strongest quarter of the area, against the rest
            core = area & (up >= np.percentile(up[area], 75))
            ratio = lum[core].mean() / max(float(np.median(lum[rest])), 1e-6)
            out["tone"] = "darker" if ratio < 0.85 else "brighter" if ratio > 1.15 else None
        out["overlay"] = _overlay_png(area, light=share >= 0.35)
        return out

    def save(self, path):
        np.savez(path, w=self.w, base=self.base, thr=self.thr, min_share=self.min_share, hint_share=self.hint_share,
                 backbone=np.array(self.backbone))

    @classmethod
    def load(cls, path):
        z = np.load(path)
        return cls(z["w"], z["base"], float(z["thr"]), str(z["backbone"]), float(z["min_share"]) if "min_share" in z else 0.0,
                   float(z["hint_share"]) if "hint_share" in z else None)


SIDES = ["right", "lower right", "bottom", "lower left", "left", "upper left", "top", "upper right"]


def side_of(u, v):
    """Which edge of the picture a point is toward, in the words a technician sees on screen."""
    ang = np.degrees(np.arctan2(v - 0.5, u - 0.5)) % 360          # 0 = right, 90 = bottom (image y grows downward)
    return SIDES[int(((ang + 22.5) % 360) // 45)]


def eye_and_field(loc, dd):
    """Which eye and which field, from the measured layout.
    Eye: in a right eye the optic disc sits to the right of the macula (94% right on DeepDRiD test photos).
    Field: whichever of the two is nearer the centre; with only one found, that one if it is near the centre."""
    d, m = loc.get("disc"), loc.get("macula")
    eye = ("right" if d[0] > m[0] else "left") if (d and m) else None
    dist = {k: np.hypot(p[0] - 0.5, p[1] - 0.5) for k, p in (("disc", d), ("macula", m)) if p}
    if len(dist) == 2:
        field = min(dist, key=dist.get)
    elif len(dist) == 1 and next(iter(dist.values())) <= 2 * dd:
        field = next(iter(dist))
    else:
        field = None
    return dict(eye=eye, field=field)


def positions_text(loc, dd):
    """Measured positions as plain facts, e.g. 'the optic disc is near the centre; the macula is near the right edge'."""
    parts = []
    for name, label in [("disc", "the optic disc"), ("macula", "the macula")]:
        p = loc.get(name)
        if p is None:
            parts.append(f"{label} is not in the picture")
        elif np.hypot(p[0] - 0.5, p[1] - 0.5) <= 2 * dd:
            parts.append(f"{label} is near the centre")
        else:
            parts.append(f"{label} is near the {side_of(*p)} edge")
    return "; ".join(parts)


def aim_advice(loc, dd):
    """What is off for the view this photo is, measured, as (fact, action); None when the layout fits its view.
    Screening takes two views of each eye, one centred on the macula and one on the optic disc. So only the structure
    the view centres on belongs in the middle (within 2 disc diameters); the other sits about 2.5 disc diameters away
    and must stay well inside the picture (within 3). The view is whichever of the two is nearer the centre."""
    name = {"disc": "optic disc", "macula": "macula"}
    found = {n: loc[n] for n in ("disc", "macula") if loc.get(n)}
    if not found:
        return ("Neither the optic disc nor the macula shows clearly", "Re-aim so the view is centred on the macula or on the optic disc")
    dist = {n: np.hypot(p[0] - 0.5, p[1] - 0.5) for n, p in found.items()}
    view = min(dist, key=dist.get)
    other = "macula" if view == "disc" else "disc"
    if dist[view] > 2 * dd:
        side = side_of(*found[view])
        if other in found:
            return (f"Neither the macula nor the optic disc is centred; the {name[view]} is nearest, toward the {side}",
                    f"Re-aim so the {name[view]} moves to the middle")
        return (f"The {name[view]} sits toward the {side} and the {name[other]} is not in the picture",
                "Re-aim so the view is centred on the macula or on the optic disc")
    if other not in found:
        return (f"The {name[view]} is centred but the {name[other]} is not in the picture",
                f"Keep the {name[view]} in the middle and turn slightly toward the {name[other]} so it comes into view")
    if dist[other] > 3 * dd:      # it normally sits about 2.5 disc diameters from the centre; further out, it is leaving the view
        return (f"The {name[view]} is centred but the {name[other]} is close to the {side_of(*found[other])} edge",
                f"Re-aim slightly so the {name[other]} comes further into the picture, keeping the {name[view]} near the middle")
    return None


def aim_instruction(loc, dd):
    """A direction measured from the photo, never guessed, for the view the photo is (macula view or disc view)."""
    a = aim_advice(loc, dd)
    if a:
        return f"{a[0]}. {a[1]}, then retake."
    found = {n: loc[n] for n in ("disc", "macula") if loc.get(n)}
    view = "optic disc" if min(found, key=lambda n: np.hypot(found[n][0] - 0.5, found[n][1] - 0.5)) == "disc" else "macula"
    # the aim check says off, but the measured layout already fits the view: only a small adjustment is needed
    return f"The {view} view is close. Fine-tune so the {view} sits exactly in the middle of the picture, then retake."


# ---------- wording: Muse Spark, capped, with templates as the fallback ----------

PROMPT = """You coach a technician who takes retinal (fundus) photos for eye screening. An automatic photo-quality
check found that a photo needs a retake because it is:
{found}
{facts}The standard instruction for this photo is:
"{standard}"
Rewrite it as ONE instruction for the technician: plain words, starting with a verb, at most 25 words. Keep every action
and every measured place in it, and add nothing else. It is not something to say to the patient. Never guess a position
or a direction. Do not mention disease or diagnosis.
Reply with JSON only: {{"instruction": "..."}}"""
PHOTO_NOTE = "The photo is attached. Look at it only to say where a shadow, glare or haze is.\n"


OBSTRUCTION_ADVICE = {
    "darker": ("Dark shadow", "Ask the patient to open wide (lift the eyelid if needed) and centre the camera on the pupil, then retake."),
    "brighter": ("Glare or haze", "Wipe the lens and adjust the working distance until it clears, then retake."),
    None: ("Something is in the way", TEMPLATES["obstruction"]),
}


def obstruction_place(ob):
    """'toward the upper right of the photo', 'in the centre of the photo', 'over most of the photo', or None."""
    w = (ob or {}).get("where")
    if not w:
        return None
    return "over most of the photo" if w == "most" else "in the centre of the photo" if w == "centre" else f"toward the {w} of the photo"


def obstruction_amount(ob):
    sh = ob.get("share", 0)
    return "most of it" if sh >= 0.5 else "a large part of it" if sh >= 0.35 else "about a quarter of it" if sh >= 0.15 else "a small part of it"


def obstruction_instruction(ob):
    """A measured place plus advice for that kind of obstruction; the standard advice when nothing was marked."""
    place = obstruction_place(ob)
    if not place:
        return TEMPLATES["obstruction"]
    what, advice = OBSTRUCTION_ADVICE[ob.get("tone")]
    return f"{what} {place}. {advice}"


def obstruction_facts(ob):
    """The measured facts Muse Spark gets instead of the photo."""
    place = obstruction_place(ob)
    if not place:
        return None
    what = {"darker": "an area darker than the rest", "brighter": "an area brighter or hazier than the rest",
            None: "an area that looks covered"}[ob.get("tone")]
    return f"{what} {place}" + ("" if ob["where"] == "most" else f", covering {obstruction_amount(ob)}")


def obstruction_summary(ob):
    """For the page: 'Toward the upper right, a small part of the photo, darker than the rest'."""
    place = obstruction_place(ob)
    if not place:
        return "Not pinned down: no part of the photo stands out"
    tone = {"darker": ", darker than the rest", "brighter": ", brighter than the rest", None: ""}[ob.get("tone")]
    if ob["where"] == "most":
        return "Over most of the photo" + tone
    return place.replace(" of the photo", "").capitalize() + ", " + obstruction_amount(ob).replace(" of it", " of the photo") + tone


def template(problems, aim_text=None, obstruction=None):
    if not problems:
        return GENERIC
    parts = [aim_text if (p == "aim" and aim_text) else obstruction_instruction(obstruction) if (p == "obstruction" and obstruction)
             else TEMPLATES[p] for p in problems]
    return " ".join([t.replace(", then retake.", ".") for t in parts[:-1]] + parts[-1:])   # say "then retake" once


def _data_url(src, px):
    img = _open(src)
    img.thumbnail((px, px))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def _json_field(text, key):
    try:
        return str(json.loads(text[text.index("{"):text.rindex("}") + 1])[key]).strip() or None
    except (ValueError, KeyError, TypeError):
        return None


class Wording:
    PRICE_IN, PRICE_OUT = 1.25e-6, 4.25e-6       # muse-spark-1.3, Standard mode, $ per token

    def __init__(self, client=None, model="muse-spark-1.3", effort="minimal", max_requests=20, max_usd=0.10, px=768):
        self.client, self.model, self.effort, self.px = client, model, effort, px
        self.max_requests, self.max_usd = max_requests, max_usd
        self.n, self.spent, self.worst = 0, 0.0, 0.01  # assume a call may cost 1 cent until a dearer one is seen

    def can_call(self):
        return self.client is not None and self.n < self.max_requests and self.spent + self.worst <= self.max_usd

    def prompt(self, problems, positions=None, obstruction=None, send_photo=False, aim_text=None):
        """Exactly the text Muse Spark gets. Without send_photo, that text is all it gets."""
        found = "\n".join(f"- {DESCRIBE[p]}" for p in problems)
        facts = ""
        if positions and "aim" in problems:         # positions only matter, and only invite comment, for aim
            facts += f"Measured positions in the picture: {positions}.\n"
        if "obstruction" in problems and obstruction_facts(obstruction):
            facts += f"Measured where the obstruction is: {obstruction_facts(obstruction)}.\n"
        if send_photo:
            facts += PHOTO_NOTE
        return PROMPT.format(found=found, facts=facts, standard=template(problems, aim_text, obstruction))

    def __call__(self, src, problems, positions=None, aim_text=None, obstruction=None, send_photo=False):
        """positions: measured facts such as 'the optic disc is near the centre; the macula is near the right edge'.
        obstruction: what ObstructionMap.locate measured. src is read only when send_photo is True."""
        fallback = template(problems, aim_text, obstruction)
        if not problems or not self.can_call():
            return dict(text=fallback, source="template", cost=0.0)
        content = [{"type": "text", "text": self.prompt(problems, positions, obstruction, send_photo, aim_text)}]
        if send_photo:
            content.append({"type": "image_url", "image_url": {"url": _data_url(src, self.px)}})
        self.n += 1
        try:
            resp = self.client.chat.completions.create(
                model=self.model, extra_body={"reasoning_effort": self.effort}, messages=[{"role": "user", "content": content}])
        except Exception as e:  # network, quota, bad key: the technician still gets an instruction
            return dict(text=fallback, source=f"template (Muse Spark error: {type(e).__name__})", cost=0.0)
        cost = resp.usage.prompt_tokens * self.PRICE_IN + resp.usage.completion_tokens * self.PRICE_OUT
        self.spent += cost
        self.worst = max(self.worst, cost)
        text = _json_field(resp.choices[0].message.content or "", "instruction")
        return dict(text=text or fallback, source="muse-spark" if text else "template (unparsed reply)", cost=cost)


# ---------- the coach's note: a session's results in words ----------

VIEW_NAMES = {("right", "macula"): "right eye centred on the macula", ("right", "disc"): "right eye centred on the optic disc",
              ("left", "macula"): "left eye centred on the macula", ("left", "disc"): "left eye centred on the optic disc"}
KIND = {"darker": "dark shadow", "brighter": "glare or haze", None: "something in the way"}
HABITS = {
    "focus": "Refocus on the small vessels before every shot.",
    "aim": "Before pressing the shutter, check that the optic disc and the macula are both near the centre.",
    "darker": "Ask each patient to open wide before the shot, and centre the camera on the pupil.",
    "brighter": "Wipe the lens between patients and hold the working distance steady.",
    None: "Ask each patient to open wide, and wipe the lens between patients.",
}
NOTE_PROMPT = """You coach a technician who takes retinal (fundus) photos for eye screening. An automatic photo-quality
check found this in today's session:
{facts}
Write a short note to the technician: at most 3 sentences and 60 words. Say what went wrong most often and one concrete
habit that would prevent it. If a view has no kept photo, name it. Use only the facts above and add no numbers of your
own. It is not for the patient. Do not mention disease or diagnosis.
Reply with JSON only: {{"note": "..."}}"""


def _clean_result(r):
    """Only the fields the note reads, each from a fixed vocabulary."""
    ob = r.get("obstruction") or {}
    return dict(verdict=r.get("verdict") if r.get("verdict") in ("keep", "retake", "not_fundus", "unreadable") else "unreadable",
                problems=[p for p in (r.get("problems") or []) if p in PROBLEMS],
                where=ob.get("where") if ob.get("where") in SIDES + ["centre", "most"] else None,
                tone=ob.get("tone") if ob.get("tone") in ("darker", "brighter") else None,
                eye=r.get("eye") if r.get("eye") in ("right", "left") else None,
                field=r.get("field") if r.get("field") in ("macula", "disc") else None)


def session_facts(results):
    """A session's results as counts in plain words. This text is all Muse Spark gets for the coach's note."""
    rs = [_clean_result(r) for r in results]
    kept = [r for r in rs if r["verdict"] == "keep"]
    back = [r for r in rs if r["verdict"] == "retake"]
    away = len(rs) - len(kept) - len(back)
    lines = [f"Photos checked: {len(rs)}. Kept: {len(kept)}. Sent back for a retake: {len(back)}."
             + (f" Not regular fundus photos, or unreadable: {away}." if away else "")]
    count = Counter(p for r in back for p in r["problems"])
    tones = Counter(r["tone"] for r in back if "obstruction" in r["problems"])
    if count:
        parts = []
        for p, c in count.most_common():
            part = f"{p} {c}"
            if p == "obstruction":
                places = Counter((r["tone"], r["where"]) for r in back if "obstruction" in r["problems"] and r["where"])
                if places:
                    part += " (" + "; ".join(f"{KIND[t]} {obstruction_place(dict(where=w))} {k}" for (t, w), k in places.most_common()) + ")"
            parts.append(part)
        lines.append("Problems on the photos sent back (a photo can have more than one): " + ", ".join(parts) + ".")
    kept_views = Counter((r["eye"], r["field"]) for r in kept if r["eye"] and r["field"])
    missing = []
    if any(r["eye"] and r["field"] for r in rs):
        lines.append("Kept photos by view: " + ", ".join(f"{name} {kept_views[v]}" for v, name in VIEW_NAMES.items()) + ".")
        missing = [name for v, name in VIEW_NAMES.items() if not kept_views[v]]
    top = count.most_common(1)[0][0] if count else None
    return dict(text="\n".join("- " + l for l in lines), n=len(rs), kept=len(kept), back=len(back), top=top,
                top_count=count[top] if top else 0, top_tone=tones.most_common(1)[0][0] if tones else None, missing=missing)


def note_template(f):
    """The coach's note without Muse Spark: the same facts, in fixed sentences."""
    if not f["n"]:
        return "No photos checked yet."
    if not f["back"]:
        s = f"None of the {f['n']} photos needed a retake."
    else:
        mostly = {"darker": ", mostly dark shadows", "brighter": ", mostly glare or haze", None: ""}[f["top_tone"]]
        what = {"focus": "blur", "aim": "aim off centre", "obstruction": "something in the way"}[f["top"]]
        habit = HABITS[f["top_tone"]] if f["top"] == "obstruction" else HABITS[f["top"]]
        s = f"{f['back']} of {f['n']} photos needed a retake, most often for {what} ({f['top_count']}){mostly}. {habit}"
    if f["missing"]:
        s += " No kept photo yet for: " + "; ".join(f["missing"]) + "."
    return s


def _note(self, facts, fallback):
    """The coach's note from session_facts(...)["text"] only. A number that is not in the facts sends it back to the
    fixed summary, so the note cannot invent counts."""
    if not self.can_call():
        return dict(text=fallback, source="template", cost=0.0)
    self.n += 1
    try:
        resp = self.client.chat.completions.create(
            model=self.model, extra_body={"reasoning_effort": self.effort},
            messages=[{"role": "user", "content": [{"type": "text", "text": NOTE_PROMPT.format(facts=facts)}]}])
    except Exception as e:
        return dict(text=fallback, source=f"template (Muse Spark error: {type(e).__name__})", cost=0.0)
    cost = resp.usage.prompt_tokens * self.PRICE_IN + resp.usage.completion_tokens * self.PRICE_OUT
    self.spent += cost
    self.worst = max(self.worst, cost)
    text = _json_field(resp.choices[0].message.content or "", "note")
    if text and not set(re.findall(r"\d+", text)) <= set(re.findall(r"\d+", facts)):
        return dict(text=fallback, source="template (Muse Spark added a number)", cost=cost)
    return dict(text=text or fallback, source="muse-spark" if text else "template (unparsed reply)", cost=cost)


Wording.note = _note


In [ ]:
%%writefile /content/retake_coach/app.py
"""Retake Coach demo server.

    pip install -r requirements.txt
    export MODEL_API_KEY=...            # optional: Muse Spark writes the coach's note; without it, a fixed summary
    python app.py                       # uses models/coach_model.npz and models/locator.npz
    python app.py --watch ~/EyerInbox   # also checks every new photo that lands in that folder
    HF_TOKEN=... python app.py          # DINOv3 (gated on Hugging Face) for the obstruction map in models/obstruction_map.npz

Pages: / checks one photo at a time during a visit; /batch checks many photos at once, or follows a watched folder
(for example the folder a camera exports to over Wi-Fi).

The photo never leaves this computer: DINOv2, the heads, the disc and macula locator and the obstruction map (DINOv3)
run here, and each instruction is a standard sentence built from what they measured. At the end of a session, the
coach's note is written by Muse Spark (Meta Model API) from the session's counts in words, within the budget below.
No photo is sent.

Budget for the coach's note (environment variables, dollars at Muse Spark 1.3 Standard prices):
    MUSE_DAILY_USD      default 0.25    MUSE_DAILY_REQUESTS default 60    MUSE_TOTAL_USD default 1.00
Past any cap the note is a fixed summary of the same counts.
"""
import argparse
import base64
import datetime
import io
import os
import threading
import time
from pathlib import Path

import numpy as np
from fastapi import Body, FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import FileResponse, JSONResponse
from PIL import Image, UnidentifiedImageError

import coach_core as cc

HERE = Path(__file__).parent
MAX_BYTES = 25 * 1024 * 1024
IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}


class Budget:
    """Muse Spark spending: a fresh daily allowance, never past the total cap for this server's lifetime."""

    def __init__(self, client, daily_usd, daily_requests, total_usd):
        self.client, self.daily_usd, self.daily_requests, self.total_usd = client, daily_usd, daily_requests, total_usd
        self.spent_before_today, self.day, self.words = 0.0, None, None
        self.lock = threading.Lock()

    def today(self):
        with self.lock:
            d = datetime.date.today()
            if d != self.day:
                if self.words is not None:
                    self.spent_before_today += self.words.spent
                left = max(0.0, self.total_usd - self.spent_before_today)
                self.day = d
                self.words = cc.Wording(self.client, max_requests=self.daily_requests, max_usd=min(self.daily_usd, left))
            return self.words

    def status(self):
        w = self.today()
        return dict(muse=self.client is not None, can_call=w.can_call(), spent_today=round(w.spent, 4),
                    cap_today=round(w.max_usd, 4), requests_today=w.n, request_cap=w.max_requests,
                    spent_total=round(self.spent_before_today + w.spent, 4), cap_total=self.total_usd)


def preview(x):
    """The 448 px square the detector saw, as a small JPEG data URL for the page."""
    buf = io.BytesIO()
    Image.fromarray((x * 255).astype(np.uint8)).save(buf, "JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def read_image(data):
    """Bytes -> RGB PIL image, or None if it is not a picture we can read."""
    try:
        img = Image.open(io.BytesIO(data))
        img.load()
        return img.convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError, ValueError):
        return None


class Checker:
    """One photo in, one result out. Shared by the single-photo page, the batch page and the watched folder."""

    def __init__(self, coach, encoder, budget, omap=None, map_encoder=None):
        self.coach, self.encoder, self.budget = coach, encoder, budget
        self.omap, self.map_encoder = omap, map_encoder               # where an obstruction is (optional)
        self.lock, self.wlock = threading.Lock(), threading.Lock()   # one photo at a time through the encoders; exact note budget

    def where_obstruction(self, x, grid, exclude=None):
        """Run the obstruction map, reusing the first pass's patch tokens when the map reads the same encoder."""
        if self.omap.backbone != self.coach.backbone or grid is None:
            with self.lock:
                _, grid = self.map_encoder([x], patches=True)
        return self.omap.locate(x, grid[0], exclude)

    def __call__(self, img):
        coach = self.coach
        t0 = time.perf_counter()
        x = cc.load_square(img)
        with self.lock:
            X, grid = self.encoder([x], patches=True) if coach.locator is not None else (self.encoder([x]), None)
        v = coach.verdict(X, np.array([cc.simple_checks(x)]), patches=grid)[0]
        t1 = time.perf_counter()
        ob, hint = None, False
        if self.omap is not None and v["looks_like_fundus"] and v["retake"]:
            named = "obstruction" in v["problems"]
            # on a retake for another reason, leave out the macula, which is naturally darker (1.25 disc diameters)
            around_macula = (*v["macula"], 1.25 * v["dd"]) if (not named and v.get("macula") and v.get("dd")) else None
            try:
                ob = self.where_obstruction(x, grid, around_macula)
            except Exception as e:                          # the extra map must never stop the check itself
                print("obstruction map:", type(e).__name__, e)
            if ob is not None and not named:
                # retake for another reason: show the area only if it is larger than clear photos ever mark (95%)
                hint = bool(ob["where"]) and ob["share"] > self.omap.hint_share
                ob = ob if hint else None
        tm = time.perf_counter()

        if not v["looks_like_fundus"]:
            verdict, words = "not_fundus", dict(text=cc.NOT_FUNDUS, source="template", cost=0.0)
        elif v["retake"]:                                   # a standard sentence built from what was measured
            named = v["problems"] + (["obstruction"] if hint else [])
            verdict, words = "retake", dict(text=cc.template(named, v.get("aim_text"), ob), source="template", cost=0.0)
        else:
            verdict, words = "keep", dict(text=cc.keep_text(v["minor"], v.get("aim_text")), source="template", cost=0.0)
        t2 = time.perf_counter()

        checks = {k: dict(score=round(v["scores"][k], 3), threshold=round(float(t), 3), found=k in v["problems"],
                          minor=k in v["minor"], marked=(k == "obstruction" and hint))
                  for k, t in zip(cc.PROBLEMS, coach.thr[1:])}
        return dict(
            verdict=verdict, instruction=words["text"], source=words["source"], cost=round(words["cost"], 5),
            problems=v["problems"], checks=checks, p_not_good=round(v["p_not_good"], 3),
            retake_threshold=round(float(coach.thr[0]), 3), rule=coach.rule, energy=round(v["energy"], 2),
            energy_limit=round(float(coach.g_thr), 2) if coach.g_thr is not None else None,
            disc=v.get("disc"), macula=v.get("macula"), dd=v.get("dd"), positions=v.get("positions"),
            eye=v.get("eye"), field=v.get("field"),
            obstruction=None if ob is None else dict(where=ob["where"], tone=ob["tone"], share=round(ob["share"], 3), hint=hint,
                                                     summary=cc.obstruction_summary(ob), overlay=ob["overlay"]),
            ms=dict(detect=round(1000 * (t1 - t0)), map=round(1000 * (tm - t1)), wording=round(1000 * (t2 - tm))),
            preview=preview(x), budget=self.budget.status())


class Watcher:
    """Checks every new photo that appears in a folder, e.g. where a camera exports over Wi-Fi (FTP or a shared folder).
    A file is checked once its size has stopped changing, so half-copied photos are not read."""

    def __init__(self, folder, checker, every=2.0):
        self.folder, self.checker, self.every = Path(folder).expanduser(), checker, every
        self.results, self.seen, self.sizes, self.seq = [], set(), {}, 0
        self.lock, self.stopped = threading.Lock(), False

    def scan_once(self):
        for p in sorted(self.folder.rglob("*")):
            if p.suffix.lower() not in IMAGE_TYPES or not p.is_file():
                continue
            key = (str(p), p.stat().st_mtime)
            if key in self.seen:
                continue
            size = p.stat().st_size
            if self.sizes.get(str(p)) != size:              # still being written, or first sight: look again next round
                self.sizes[str(p)] = size
                continue
            self.seen.add(key)
            img = read_image(p.read_bytes())
            r = dict(verdict="unreadable", instruction="Not a photo the check can read.") if img is None else self.checker(img)
            r.update(name=str(p.relative_to(self.folder)), arrived=datetime.datetime.now().strftime("%H:%M:%S"))
            with self.lock:
                r["seq"] = self.seq                         # a running number, so pages can ask for what is new
                self.seq += 1
                self.results.append(r)

    def run(self):
        while not self.stopped:
            try:
                if self.folder.is_dir():
                    self.scan_once()
            except Exception as e:                          # keep watching whatever happens to one file
                print("watch:", type(e).__name__, e)
            time.sleep(self.every)

    def start(self):
        threading.Thread(target=self.run, daemon=True).start()
        return self

    def stop(self):
        self.stopped = True

    def since(self, n):
        with self.lock:
            return [r for r in self.results if r["seq"] >= n]

    def clear(self):
        """Forget the results shown so far; files already checked are not checked again."""
        with self.lock:
            self.results = []


class Store:
    """What the two pages have checked so far, kept here so that switching pages or reloading loses nothing:
    the current visit on the one-photo page, and the list on the many-photos page. In memory only; a restart clears it."""

    def __init__(self, limit=2000):
        self.lists, self.lock, self.limit = {"visit": [], "batch": []}, threading.Lock(), limit

    def add(self, which, r):
        with self.lock:
            self.lists[which].append(r)
            del self.lists[which][:-self.limit]

    def clear(self, which):
        with self.lock:
            self.lists[which] = []

    def get(self):
        with self.lock:
            return {k: list(v) for k, v in self.lists.items()}


def create_app(coach, encoder, budget, examples_dir=None, watcher=None, checker=None, store=None):
    app = FastAPI(title="Retake Coach")
    checker = checker or (watcher.checker if watcher else Checker(coach, encoder, budget))
    store = store or Store()
    examples = Path(examples_dir) if examples_dir else None

    @app.get("/")
    def index():
        return FileResponse(HERE / "static" / "index.html")

    @app.get("/batch")
    def batch():
        return FileResponse(HERE / "static" / "batch.html")

    @app.get("/api/status")
    def status():
        return dict(budget=budget.status(), thresholds=dict(zip(cc.Coach.HEADS, map(float, coach.thr))),
                    energy_limit=coach.g_thr, watching=str(watcher.folder) if watcher else None,
                    detector=coach.backbone, obstruction_map=checker.omap.backbone if checker.omap else None)

    @app.get("/api/examples")
    def list_examples():
        """Sample photos, with an optional note per file from examples/labels.json (e.g. the dataset's own label)."""
        if not examples or not examples.is_dir():
            return []
        notes = {}
        if (examples / "labels.json").is_file():
            import json
            notes = json.loads((examples / "labels.json").read_text())
        return [dict(name=p.name, note=notes.get(p.name, "")) for p in sorted(examples.iterdir()) if p.suffix.lower() in IMAGE_TYPES]

    @app.get("/examples/{name}")
    def example(name: str):
        p = (examples / name).resolve() if examples else None
        if not p or p.parent != examples.resolve() or not p.is_file():
            raise HTTPException(404, "No such sample photo.")
        return FileResponse(p)

    @app.post("/api/check")
    def check(photo: UploadFile = File(...), keep_in: str = Form(""), name: str = Form(""),
              sample_name: str = Form(""), sample_note: str = Form("")):
        """keep_in: 'visit' or 'batch' keeps the result for that page (see /api/state)."""
        data = photo.file.read(MAX_BYTES + 1)
        if len(data) > MAX_BYTES:
            raise HTTPException(413, "That file is over 25 MB. Export the photo as a JPEG and try again.")
        img = read_image(data)
        if img is None:
            raise HTTPException(400, "That file is not a photo the check can read. Use a JPEG or PNG.")
        r = checker(img)
        r.update(name=name or photo.filename or "photo", arrived=datetime.datetime.now().strftime("%H:%M:%S"),
                 sample=dict(name=sample_name, text=sample_note) if sample_name else None)
        if keep_in in ("visit", "batch"):
            store.add(keep_in, r)
        return JSONResponse(r)

    @app.get("/api/state")
    def state():
        """Everything checked so far on each page, oldest first."""
        return store.get()

    @app.post("/api/clear")
    def clear(which: str):
        """'visit' (New patient) or 'batch' (Clear list; also forgets photos shown from the watched folder)."""
        if which not in ("visit", "batch"):
            raise HTTPException(400, "Clear 'visit' or 'batch'.")
        store.clear(which)
        if which == "batch" and watcher:
            watcher.clear()
        return dict(cleared=which)

    @app.post("/api/note")
    def coach_note(payload: dict = Body(...)):
        """The coach's note for a session. The page sends each result's verdict, problems, obstruction place and tone,
        eye and field; only those are read, and only their counts in words go to Muse Spark."""
        results = payload.get("results")
        if not isinstance(results, list) or len(results) > 5000:
            raise HTTPException(400, "Send the session's results as a list.")
        facts = cc.session_facts([r for r in results if isinstance(r, dict)])
        fallback = cc.note_template(facts)
        if payload.get("use_muse", True) and facts["n"]:
            with checker.wlock:
                out = budget.today().note(facts["text"], fallback)
        else:
            out = dict(text=fallback, source="template", cost=0.0)
        return dict(note=out["text"], source=out["source"], cost=round(out["cost"], 5), facts=facts["text"], budget=budget.status())

    @app.get("/api/watch")
    def watch(since: int = 0):
        if not watcher:
            return dict(folder=None, results=[])
        return dict(folder=str(watcher.folder), results=watcher.since(since))

    return app


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="models/coach_model.npz")
    ap.add_argument("--locator", default="models/locator.npz", help="optional: disc and macula positions for aim directions")
    ap.add_argument("--map", default="models/obstruction_map.npz", help="optional: where an obstruction is (DINOv3; needs HF_TOKEN)")
    ap.add_argument("--examples", default="examples")
    ap.add_argument("--watch", default=None, help="folder to check new photos in as they arrive")
    ap.add_argument("--host", default="127.0.0.1")
    ap.add_argument("--port", type=int, default=8000)
    a = ap.parse_args()

    client = None
    if os.environ.get("MODEL_API_KEY"):
        from openai import OpenAI
        client = OpenAI(base_url="https://api.meta.ai/v1", api_key=os.environ["MODEL_API_KEY"], timeout=30)
    budget = Budget(client, float(os.environ.get("MUSE_DAILY_USD", 0.25)), int(os.environ.get("MUSE_DAILY_REQUESTS", 60)),
                    float(os.environ.get("MUSE_TOTAL_USD", 1.00)))
    coach = cc.Coach.load(a.model)
    if os.path.exists(a.locator):
        coach.attach_locator(cc.Locator.load(a.locator))
    print(f"loading {coach.backbone} ...")
    encoder = cc.Encoder(coach.backbone)
    omap = map_encoder = None
    if os.path.exists(a.map):
        omap = cc.ObstructionMap.load(a.map)
        try:
            map_encoder = encoder if omap.backbone == coach.backbone else cc.Encoder(omap.backbone)
        except Exception as e:                               # DINOv3 is gated: without access the page works without the map
            print(f"obstruction map off: could not load {omap.backbone} ({type(e).__name__}). Set HF_TOKEN after accepting its licence.")
            omap = None
    checker = Checker(coach, encoder, budget, omap, map_encoder)
    watcher = Watcher(a.watch, checker).start() if a.watch else None
    app = create_app(coach, encoder, budget, a.examples, watcher, checker)
    import uvicorn
    uvicorn.run(app, host=a.host, port=a.port)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/retake_coach/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Retake Coach</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible+Next:wght@400;600;800&display=swap" rel="stylesheet">
<style>
/* Dark on purpose: fundus cameras are used in dimmed rooms, so the screen must not light the room up. */
:root {
  --room: #171c22;      /* exam-room slate */
  --panel: #202830;
  --line: #34414d;
  --ink: #e6ebf0;
  --ink-2: #9eacba;
  --keep: #63c29e;      /* the camera's green fixation light */
  --retake: #f4b740;    /* amber "not ready" lamp */
  --away: #aab5c2;
  --focus: #8fc1ff;
  color-scheme: dark;
}
* { box-sizing: border-box; }
html, body { margin: 0; background: var(--room); color: var(--ink); }
body {
  font-family: "Atkinson Hyperlegible Next", "Atkinson Hyperlegible", system-ui, sans-serif;
  font-size: 17px; line-height: 1.5;
  -webkit-font-smoothing: antialiased;
}
:focus-visible { outline: 3px solid var(--focus); outline-offset: 3px; border-radius: 6px; }
.sr { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }

header {
  display: flex; align-items: center; justify-content: space-between; gap: 16px; flex-wrap: wrap;
  padding: 18px clamp(16px, 4vw, 40px); border-bottom: 1px solid var(--line);
}
.mark { display: flex; align-items: center; gap: 10px; font-weight: 800; font-size: 21px; letter-spacing: -0.01em; }
.mark svg { width: 26px; height: 26px; }
.left { display: flex; align-items: center; gap: 28px; flex-wrap: wrap; }
nav { display: flex; gap: 6px; font-size: 15px; }
nav a { color: var(--ink-2); text-decoration: none; padding: 4px 2px; margin: 0 8px 0 0; border-bottom: 2px solid transparent; }
nav a:hover { color: var(--ink); }
nav a[aria-current] { color: var(--ink); border-bottom-color: var(--keep); }
.status { display: flex; gap: 8px; flex-wrap: wrap; font-size: 14px; color: var(--ink-2); }
.status span { border: 1px solid var(--line); border-radius: 999px; padding: 4px 12px; }
.status b { color: var(--ink); font-weight: 600; }

main {
  display: grid; grid-template-columns: minmax(280px, 540px) minmax(0, 560px); column-gap: clamp(24px, 5vw, 72px);
  row-gap: 24px; grid-template-areas: "photo result" "samples result" "session session";
  justify-content: center; align-items: start; padding: clamp(20px, 4vw, 48px) clamp(16px, 4vw, 40px);
}
.photo { grid-area: photo; } .result { grid-area: result; } .samples { grid-area: samples; } .session { grid-area: session; }
@media (max-width: 900px) {
  main { grid-template-columns: minmax(0, 520px); grid-template-areas: "photo" "result" "samples" "session"; }
}

/* ---- viewfinder: the photo in its circle, ringed by the three checks ---- */
.viewfinder { position: relative; aspect-ratio: 660 / 520; width: 100%; }
.viewfinder svg { width: 100%; height: 100%; display: block; overflow: visible; }
.arc { fill: none; stroke: var(--line); stroke-width: 4; stroke-linecap: round; }
.arc.clear { stroke: var(--keep); }
.arc.found { stroke: var(--retake); stroke-width: 12; }
.arc.minor { stroke: var(--retake); stroke-width: 4; }
.arc.off { stroke: var(--away); stroke-dasharray: 3 9; }
.arc-label { fill: var(--ink-2); font-size: 26px; font-weight: 600; }
.arc-label.found { fill: var(--retake); }
.rim { fill: none; stroke: var(--line); stroke-width: 2; stroke-dasharray: 6 8; }
.has-photo .rim { stroke-dasharray: none; stroke: #0000; }
.pos-disc { fill: none; stroke: var(--ink); stroke-width: 3; }
.pos-macula { fill: var(--ink); stroke: var(--room); stroke-width: 2; }
.pos-zone { fill: none; stroke: var(--ink); stroke-width: 1.5; stroke-dasharray: 5 7; opacity: 0.8; }
.pos-label { fill: var(--ink); font-size: 18px; font-weight: 600; paint-order: stroke; stroke: var(--room); stroke-width: 4px; }
.scan { fill: none; stroke: var(--focus); stroke-width: 4; stroke-linecap: round; opacity: 0; transform-box: fill-box; transform-origin: center; }
.checking .scan { opacity: 1; animation: spin 1.1s linear infinite; }
@keyframes spin { to { transform: rotate(360deg); } }
.drawn .arc { animation: draw 0.7s ease-out both; }
@keyframes draw { from { stroke-dashoffset: var(--len); } to { stroke-dashoffset: 0; } }
@media (prefers-reduced-motion: reduce) {
  .checking .scan { animation: none; stroke-dasharray: 40 1000; }
  .drawn .arc { animation: none; }
}

.drop {
  position: absolute; inset: 11.5% 19.7%; border-radius: 50%; display: grid; place-content: center; gap: 14px;
  text-align: center; cursor: pointer; padding: 12%;
}
.drop .lead { font-size: 21px; font-weight: 600; }
.drop .btn {
  justify-self: center; background: var(--ink); color: var(--room); font-weight: 800; border-radius: 999px; padding: 10px 22px;
}
.has-photo .drop { opacity: 0; background: rgba(23, 28, 34, 0.78); transition: opacity 0.15s; }
.has-photo .drop:hover, .has-photo .drop:focus-within, .dragging .drop { opacity: 1; }
.dragging .rim { stroke: var(--focus); stroke-dasharray: none; }
.error { color: var(--retake); min-height: 1.5em; margin: 12px 0 0; text-align: center; }

.samples h2, .session h2, .result h3 { font-size: 15px; font-weight: 600; color: var(--ink-2); margin: 0 0 10px; }
.thumbs { display: flex; gap: 10px; flex-wrap: wrap; }
.thumbs button {
  width: 56px; height: 56px; padding: 0; border-radius: 50%; overflow: hidden; border: 2px solid var(--line);
  background: #000; cursor: pointer;
}
.thumbs button:hover { border-color: var(--ink-2); }
.thumbs img { width: 100%; height: 100%; object-fit: cover; display: block; }
.thumbs .keep { border-color: var(--keep); }
.thumbs .retake { border-color: var(--retake); }
.thumbs .not_fundus { border-color: var(--away); border-style: dashed; }

/* ---- result ---- */
.result { min-height: 200px; }
.idle h1 { font-size: 33px; line-height: 1.2; margin: 8px 0 12px; font-weight: 800; letter-spacing: -0.015em; }
.idle p { color: var(--ink-2); max-width: 46ch; margin: 0 0 12px; }
.verdict { display: flex; align-items: center; gap: 10px; font-size: 26px; font-weight: 800; margin: 4px 0 6px; }
.verdict svg { width: 26px; height: 26px; flex: none; }
.v-keep .verdict { color: var(--keep); }
.v-retake .verdict { color: var(--retake); }
.v-not_fundus .verdict { color: var(--away); }
.instruction { font-size: clamp(26px, 3.2vw, 34px); line-height: 1.22; font-weight: 800; letter-spacing: -0.012em; margin: 0 0 10px; max-width: 24ch; }
.source { color: var(--ink-2); font-size: 14px; margin: 0 0 28px; }
.rows { margin: 0; display: grid; grid-template-columns: max-content 1fr; gap: 10px 18px; align-items: baseline; }
.rows dt { color: var(--ink-2); }
.rows dd { margin: 0; }
.rows .found { color: var(--retake); font-weight: 600; }
.meter { position: relative; height: 6px; border-radius: 3px; background: var(--line); margin-top: 6px; max-width: 260px; }
.meter i { position: absolute; inset: 0 auto 0 0; border-radius: 3px; background: var(--ink-2); }
.meter.found i { background: var(--retake); }
.meter b { position: absolute; top: -4px; width: 2px; height: 14px; background: var(--ink); }
.small { font-size: 14px; color: var(--ink-2); }
.toggle { display: flex; gap: 10px; align-items: flex-start; margin-top: 26px; font-size: 15px; color: var(--ink-2); max-width: 52ch; }
.toggle input { width: 18px; height: 18px; margin-top: 3px; accent-color: var(--keep); }

.session { border-top: 1px solid var(--line); padding-top: 18px; }
.session .count { color: var(--ink-2); font-size: 15px; margin: 14px 0 12px; }
.visit-head { display: flex; align-items: baseline; justify-content: space-between; gap: 16px; }
.ghost { background: none; color: var(--ink); border: 1px solid var(--line); border-radius: 999px; padding: 6px 14px; font: inherit; font-size: 14px; cursor: pointer; }
.ghost:hover { border-color: var(--ink-2); }
.visit-actions { display: flex; gap: 8px; flex-wrap: wrap; }
.note { margin: 16px 0 6px; background: var(--panel); border-left: 4px solid var(--keep); border-radius: 0 10px 10px 0; padding: 12px 16px; max-width: 72ch; }
.note p { margin: 0; }
.note .src { color: var(--ink-2); font-size: 14px; margin-top: 6px; }
.note details { margin-top: 6px; font-size: 14px; color: var(--ink-2); }
.note summary { cursor: pointer; }
.note pre { white-space: pre-wrap; font: inherit; margin: 6px 0 0; }

.visit { border-collapse: collapse; font-size: 15px; }
.visit th { font-weight: 600; color: var(--ink-2); text-align: left; padding: 6px 18px 6px 0; }
.visit td { padding: 6px 18px 6px 0; }
.visit .kept { color: var(--keep); font-weight: 600; }
.visit .retake { color: var(--retake); font-weight: 600; }
.visit .none { color: var(--ink-2); }
footer { padding: 8px clamp(16px, 4vw, 40px) 32px; color: var(--ink-2); font-size: 14px; max-width: 1150px; margin: 0 auto; }
footer p { margin: 4px 0; }
</style>
</head>
<body>
<header>
  <div class="left">
    <div class="mark">
      <svg viewBox="0 0 26 26" aria-hidden="true"><circle cx="13" cy="13" r="11" fill="none" stroke="#63c29e" stroke-width="2.4"/>
        <circle cx="13" cy="13" r="4" fill="#f4b740"/></svg>
      Retake Coach
    </div>
    <nav aria-label="Pages"><a href="/" aria-current="page">One photo</a><a href="/batch">Many photos</a></nav>
  </div>
  <div class="status" aria-live="polite">
    <span id="detector-status">Detector: <b>DINOv2 on this computer</b></span>
    <span id="wording-status">Coach's note: <b>checking…</b></span>
  </div>
</header>

<main>
  <section class="photo" aria-label="Photo">
    <div class="viewfinder" id="vf">
      <svg viewBox="-80 -10 660 520" aria-hidden="true">
        <defs><clipPath id="disc"><circle cx="250" cy="250" r="200"/></clipPath></defs>
        <circle class="rim" cx="250" cy="250" r="200"/>
        <image id="photo" x="50" y="50" width="400" height="400" clip-path="url(#disc)" preserveAspectRatio="xMidYMid slice"/>
        <image id="obstruction-layer" x="50" y="50" width="400" height="400" clip-path="url(#disc)" preserveAspectRatio="none"/>
        <g id="arcs"></g>
        <g id="positions"></g>
        <circle class="scan" cx="250" cy="250" r="222" stroke-dasharray="90 1300"/>
      </svg>
      <label class="drop" for="file">
        <span class="lead" id="drop-lead">Drop the photo you just took</span>
        <span class="btn">Choose a photo</span>
      </label>
      <input class="sr" type="file" id="file" accept="image/*">
    </div>
    <p class="error" id="error" role="alert"></p>
  </section>
  <section class="samples" id="samples" hidden>
    <h2>Sample photos</h2>
    <div class="thumbs" id="sample-thumbs"></div>
  </section>

  <section class="result idle" id="result" aria-live="polite">
    <h1>Check each fundus photo before the patient leaves the chair.</h1>
    <p>Retake Coach says whether the photo is good enough to grade. If it isn't, it tells you what to fix: focus, aim, or something blocking the view.</p>
    <p>It checks photo quality only. It does not look for disease.</p>
  </section>

  <section class="session" id="session" hidden>
    <div class="visit-head"><h2>This visit</h2><div class="visit-actions"><button class="ghost" id="note-btn" type="button">Coach's note</button><button class="ghost" id="new-patient" type="button">New patient</button></div></div>
    <table class="visit" id="visit" aria-label="Photos this visit needs"></table>
    <p class="count" id="session-count"></p>
    <div class="note" id="note" aria-live="polite" hidden></div>
    <div class="thumbs" id="session-thumbs"></div>
  </section>
</main>

<footer>
  <p>Detector: Meta's DINOv2 image features with small heads trained on DeepDRiD photo-quality labels (CC BY-SA 4.0). Where an obstruction is: DINOv3 patch features, when installed. Both run on this computer, and photos stay on it.</p>
  <p>Instructions are standard sentences built from what was measured. The coach's note is written by Muse Spark through the Meta Model API from the visit's counts in words, never photos, under a daily spending cap; without it, a standard summary is shown.</p>
</footer>

<script>
const PROBLEMS = { focus: [-56, 56], aim: [64, 176], obstruction: [184, 296] };
const WORDS = {
  focus: ['Blurry', 'Sharp'], aim: ['Off centre', 'Centred'], obstruction: ['Shadow or glare', 'Clear view']
};
const NAMES = { focus: 'Focus', aim: 'Aim', obstruction: 'Obstruction' };
const ICONS = {
  keep: '<svg viewBox="0 0 26 26"><circle cx="13" cy="13" r="11" fill="none" stroke="currentColor" stroke-width="2.6"/><path d="M8 13.5l3.4 3.4L18.5 9" fill="none" stroke="currentColor" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/></svg>',
  retake: '<svg viewBox="0 0 26 26"><path d="M13 2.5l11 19.5H2z" fill="none" stroke="currentColor" stroke-width="2.4" stroke-linejoin="round"/><path d="M13 10v5.5" stroke="currentColor" stroke-width="2.6" stroke-linecap="round"/><circle cx="13" cy="18.8" r="1.5" fill="currentColor"/></svg>',
  not_fundus: '<svg viewBox="0 0 26 26"><circle cx="13" cy="13" r="11" fill="none" stroke="currentColor" stroke-width="2.4" stroke-dasharray="4 3"/><path d="M9 9l8 8M17 9l-8 8" stroke="currentColor" stroke-width="2.4" stroke-linecap="round"/></svg>'
};
const HEAD = { keep: 'Keep', retake: 'Retake', not_fundus: 'Not a fundus photo' };
const $ = id => document.getElementById(id);
const vf = $('vf'), result = $('result');
let session = [];

function polar(r, deg) { const a = (deg - 90) * Math.PI / 180; return [250 + r * Math.cos(a), 250 + r * Math.sin(a)]; }
function arcPath(r, a0, a1) {
  const [x0, y0] = polar(r, a0), [x1, y1] = polar(r, a1);
  return `M${x0.toFixed(1)} ${y0.toFixed(1)} A${r} ${r} 0 ${a1 - a0 > 180 ? 1 : 0} 1 ${x1.toFixed(1)} ${y1.toFixed(1)}`;
}
function drawArcs(states) {           // states: {focus: 'found'|'clear'|'off'|''}
  const g = $('arcs'); g.innerHTML = '';
  for (const [k, [a0, a1]] of Object.entries(PROBLEMS)) {
    const len = (Math.PI * 222 * (a1 - a0) / 180).toFixed(0);
    const p = document.createElementNS('http://www.w3.org/2000/svg', 'path');
    p.setAttribute('d', arcPath(222, a0, a1));
    p.setAttribute('class', 'arc ' + (states[k] || ''));
    p.style.setProperty('--len', len); p.style.strokeDasharray = states[k] === 'off' ? '' : `${len} ${len}`;
    g.appendChild(p);
    const mid = (a0 + a1) / 2, [x, y] = polar(250, mid);
    const t = document.createElementNS('http://www.w3.org/2000/svg', 'text');
    t.setAttribute('x', x.toFixed(1)); t.setAttribute('y', (y + 6).toFixed(1));
    t.setAttribute('text-anchor', Math.abs(Math.sin(mid * Math.PI / 180)) < 0.3 ? 'middle' : (x > 250 ? 'start' : 'end'));
    t.setAttribute('class', 'arc-label ' + (states[k] === 'found' ? 'found' : ''));
    t.textContent = NAMES[k];
    g.appendChild(t);
  }
}
drawArcs({});

function drawPositions(r) {          // measured optic disc (ring) and macula (dot); the 2-disc-diameter zone when aim is off
  const g = $('positions'); g.innerHTML = '';
  if (r.verdict === 'not_fundus' || !r.dd) return;
  const ns = 'http://www.w3.org/2000/svg', at = ([u, v]) => [50 + 400 * u, 50 + 400 * v];
  const add = (tag, attrs) => { const e = document.createElementNS(ns, tag); for (const k in attrs) e.setAttribute(k, attrs[k]); g.appendChild(e); return e; };
  if (r.checks.aim.found || r.checks.aim.minor) add('circle', { cx: 250, cy: 250, r: 800 * r.dd, class: 'pos-zone' });
  if (r.disc) { const [x, y] = at(r.disc); add('circle', { cx: x, cy: y, r: 200 * r.dd, class: 'pos-disc' }); }
  if (r.macula) { const [x, y] = at(r.macula); add('circle', { cx: x, cy: y, r: 7, class: 'pos-macula' }); }
}

function cap(s) { return s[0].toUpperCase() + s.slice(1); }

function drawVisit() {                 // the four views a screening visit needs, from the photos checked so far
  const cell = {};
  for (const { r } of session) {
    if (r.verdict === 'not_fundus' || !r.eye || !r.field) continue;
    const k = r.eye + '|' + r.field;
    if (r.verdict === 'keep') cell[k] = 'kept'; else if (cell[k] !== 'kept') cell[k] = 'retake';
  }
  const word = { kept: 'Kept', retake: 'Retake needed', undefined: 'Not yet' };
  let html = '<tr><th></th><th>Centred on macula</th><th>Centred on disc</th></tr>';
  for (const eye of ['right', 'left']) {
    html += `<tr><th>${cap(eye)} eye</th>` + ['macula', 'disc'].map(f => {
      const st = cell[eye + '|' + f];
      return `<td class="${st || 'none'}">${word[st]}</td>`;
    }).join('') + '</tr>';
  }
  $('visit').innerHTML = html;
  return Object.values(cell).filter(v => v === 'kept').length;
}

function esc(s) { return String(s).replace(/[&<>"]/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;' }[c])); }
function money(x) { return x === 0 ? '$0' : '$' + (x < 0.01 ? x.toFixed(4) : x.toFixed(2)); }

function setStatus(b) {
  const el = $('wording-status');
  if (!b.muse) el.innerHTML = "Coach's note: <b>standard (no Muse Spark key)</b>";
  else if (!b.can_call) el.innerHTML = "Coach's note: <b>standard (today's Muse Spark cap reached)</b>";
  else el.innerHTML = `Coach's note: <b>Muse Spark</b>, ${money(b.spent_today)} of ${money(b.cap_today)} used today`;
}

function render(r, note) {
  vf.classList.remove('checking'); vf.classList.add('has-photo');
  $('photo').setAttribute('href', r.preview);
  $('drop-lead').textContent = 'Drop or choose another photo';
  const states = {};
  for (const k of Object.keys(PROBLEMS)) states[k] = r.verdict === 'not_fundus' ? 'off' : (r.checks[k].found ? 'found' : (r.checks[k].minor || r.checks[k].marked) ? 'minor' : 'clear');
  vf.classList.remove('drawn'); void vf.offsetWidth; vf.classList.add('drawn');
  drawArcs(states);
  drawPositions(r);
  const layer = $('obstruction-layer');
  if (r.obstruction && r.obstruction.overlay) layer.setAttribute('href', r.obstruction.overlay); else layer.removeAttribute('href');

  let rows = '';
  if (r.verdict !== 'not_fundus') {
    for (const k of Object.keys(PROBLEMS)) {
      const c = r.checks[k], found = c.found, flagged = c.found || c.minor || c.marked;
      rows += `<dt>${NAMES[k]}</dt><dd><span class="${found ? 'found' : ''}">${WORDS[k][flagged ? 0 : 1]}${c.minor ? ' (still gradable)' : c.marked ? ' (from the marked area)' : ''}</span>
        <span class="small">score ${c.score.toFixed(2)}, flagged from ${c.threshold.toFixed(2)}</span>
        <div class="meter ${found ? 'found' : ''}" aria-hidden="true"><i style="width:${(100 * c.score).toFixed(0)}%"></i><b style="left:${(100 * c.threshold).toFixed(0)}%"></b></div></dd>`;
    }
    rows += r.rule === 'problems'
      ? `<dt>Overall</dt><dd>Quality score ${r.p_not_good.toFixed(2)} <span class="small">for reference; a retake is asked only for a problem above</span></dd>`
      : `<dt>Overall</dt><dd>${r.p_not_good >= r.retake_threshold ? 'Not good enough to grade' : 'Good enough to grade'}
      <span class="small">score ${r.p_not_good.toFixed(2)}, retake from ${r.retake_threshold.toFixed(2)}</span></dd>`;
    if (r.obstruction) rows += `<dt>Where</dt><dd>${esc(r.obstruction.summary)}
      <span class="small">${!r.obstruction.where ? 'no single area stood out enough to mark'
        : r.obstruction.hint ? 'blue on the photo. The obstruction score stayed under its line, but clear photos rarely mark this much'
        : 'blue on the photo, measured by the obstruction check itself'}</span></dd>`;
  }
  rows += `<dt>Fundus photo</dt><dd>${r.verdict === 'not_fundus' ? 'No, unlike the photos it knows' : 'Yes'}
    <span class="small">energy ${r.energy.toFixed(1)}, turned away above ${r.energy_limit.toFixed(1)}</span></dd>
    <dt>Time</dt><dd>Checked in ${r.ms.detect} ms${r.ms.map > 0 ? `, located in ${r.ms.map} ms` : ''}${r.ms.wording > 50 ? `, instruction in ${(r.ms.wording / 1000).toFixed(1)} s` : ''}</dd>`;
  if (r.positions && r.verdict !== 'not_fundus' && (r.checks.aim.found || r.checks.aim.minor)) rows += `<dt>Positions</dt><dd>${esc(r.positions[0].toUpperCase() + r.positions.slice(1))}
    <span class="small">measured on the photo: ring = optic disc, dot = macula</span></dd>`;
  if (r.verdict !== 'not_fundus') rows += `<dt>Photo</dt><dd>${r.eye ? cap(r.eye) + ' eye' : 'Eye not measured'}${r.field ? `, centred on the ${r.field === 'disc' ? 'optic disc' : 'macula'}` : ''}
    <span class="small">${r.eye ? 'from where the disc sits next to the macula' : 'both the disc and the macula are needed to tell'}</span></dd>`;
  if (note) rows += `<dt>Sample</dt><dd>${esc(note.name)} ${note.text ? `<span class="small">${esc(note.text)}</span>` : ''}</dd>`;

  const src = r.verdict === 'retake' ? 'Built from what was measured on this photo' : '';
  result.className = 'result v-' + r.verdict;
  result.innerHTML = `
    <div class="verdict">${ICONS[r.verdict]}<span>${HEAD[r.verdict]}</span></div>
    <p class="instruction">${esc(r.instruction)}</p>
    <p class="source">${src}</p>
    <h3>What the check saw</h3>
    <dl class="rows">${rows}</dl>`;
  setStatus(r.budget);
}

function addToSession(r, note) {
  session.unshift({ r, note });
  $('session').hidden = false; $('note').hidden = true;
  const n = session.length, re = session.filter(s => s.r.verdict === 'retake').length;
  const away = session.filter(s => s.r.verdict === 'not_fundus').length;
  const done = drawVisit();
  const unknown = session.filter(s => s.r.verdict !== 'not_fundus' && (!s.r.eye || !s.r.field)).length;
  $('session-count').textContent = `${done} of 4 views kept. ${n} checked, ${n - re - away} kept, ${re} to retake`
    + (away ? `, ${away} not ${away === 1 ? 'a fundus photo' : 'fundus photos'}` : '')
    + (unknown ? `; ${unknown} where the eye or field couldn't be measured` : '') + '.';
  const box = $('session-thumbs'); box.innerHTML = '';
  session.forEach((s, i) => {
    const b = document.createElement('button');
    b.className = s.r.verdict; b.title = HEAD[s.r.verdict];
    b.setAttribute('aria-label', `Photo ${n - i}: ${HEAD[s.r.verdict]}`);
    b.innerHTML = `<img alt="" src="${s.r.preview}">`;
    b.addEventListener('click', () => render(s.r, s.note));
    box.appendChild(b);
  });
}

async function check(blob, note) {
  $('error').textContent = '';
  vf.classList.add('checking');
  if (!vf.classList.contains('has-photo')) {
    vf.classList.add('has-photo'); $('photo').setAttribute('href', URL.createObjectURL(blob));
  }
  const fd = new FormData(); fd.append('photo', blob, blob.name || 'photo.jpg'); fd.append('keep_in', 'visit');   // kept on the server for this visit
  if (note) { fd.append('sample_name', note.name); fd.append('sample_note', note.text || ''); }
  try {
    const res = await fetch('/api/check', { method: 'POST', body: fd });
    const body = await res.json().catch(() => ({}));
    if (!res.ok) throw new Error(body.detail || 'The check did not finish. Try the photo again.');
    render(body, note); addToSession(body, note);
  } catch (e) {
    vf.classList.remove('checking');
    if (!session.length) { vf.classList.remove('has-photo'); $('photo').removeAttribute('href'); }
    $('error').textContent = e.message.includes('fetch') ? 'The check server is not reachable. Make sure app.py is running.' : e.message;
  }
}


// The coach's note: only each result's verdict, problems, obstruction place and tone, eye and field are sent,
// and the server passes only their counts in words to Muse Spark.
const slim = r => ({ verdict: r.verdict, problems: r.problems || [], eye: r.eye || null, field: r.field || null,
                     obstruction: r.obstruction ? { where: r.obstruction.where, tone: r.obstruction.tone } : null });
async function coachNote(results, box, btn) {
  const label = btn.textContent; btn.disabled = true; btn.textContent = 'Writing…';
  try {
    const res = await fetch('/api/note', { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ results: results.map(slim) }) });
    const b = await res.json();
    if (!res.ok) throw new Error(b.detail || 'failed');
    const src = b.source === 'muse-spark' ? `Written by Muse Spark for ${money(b.cost)}. No photos were sent.`
              : `Standard summary${b.source.startsWith('template (') ? ` (${esc(b.source.slice(10, -1))})` : ''}.`;
    box.innerHTML = `<p>${esc(b.note)}</p><p class="src">${src}</p>
      <details><summary>What Muse Spark was given</summary><pre>${esc(b.facts)}</pre></details>`;
    if (b.budget) setStatus(b.budget);
  } catch (e) {
    box.innerHTML = '<p>The note could not be written. Make sure app.py is running.</p>';
  }
  box.hidden = false; btn.textContent = label; btn.disabled = false;
}

$('note-btn').addEventListener('click', () => coachNote(session.map(x => x.r), $('note'), $('note-btn')));
$('new-patient').addEventListener('click', () => {
  session = []; $('session').hidden = true; $('session-thumbs').innerHTML = ''; $('note').hidden = true;
  fetch('/api/clear?which=visit', { method: 'POST' }).catch(() => {});
});
$('file').addEventListener('change', e => { const f = e.target.files[0]; if (f) check(f, ''); e.target.value = ''; });
['dragenter', 'dragover'].forEach(t => vf.addEventListener(t, e => { e.preventDefault(); vf.classList.add('dragging'); }));
['dragleave', 'drop'].forEach(t => vf.addEventListener(t, e => { e.preventDefault(); vf.classList.remove('dragging'); }));
vf.addEventListener('drop', e => { const f = e.dataTransfer.files[0]; if (f) check(f, ''); });

// Coming back from the other page, or reloading: bring back this visit's photos and show the latest one.
fetch('/api/state').then(r => r.json()).then(st => {
  if (!st.visit.length || session.length) return;
  for (const r of st.visit) addToSession(r, r.sample);
  const last = st.visit[st.visit.length - 1]; render(last, last.sample);
}).catch(() => {}).then(() => fetch('/api/status')).then(r => r.json()).then(s => {
  setStatus(s.budget);
  if (s.obstruction_map && s.obstruction_map !== s.detector) $('detector-status').innerHTML = 'Detector: <b>DINOv2 + DINOv3 on this computer</b>';
}).catch(() => {
  $('wording-status').innerHTML = "Coach's note: <b>server not reachable</b>";
});
fetch('/api/examples').then(r => r.json()).then(list => {
  if (!list.length) return;
  $('samples').hidden = false;
  for (const ex of list) {
    const b = document.createElement('button');
    b.setAttribute('aria-label', `Check sample photo ${ex.name}`); b.title = ex.name;
    b.innerHTML = `<img alt="" loading="lazy" src="/examples/${encodeURIComponent(ex.name)}">`;
    b.addEventListener('click', async () => {
      const blob = await (await fetch('/examples/' + encodeURIComponent(ex.name))).blob();
      blob.name = ex.name; check(blob, { name: ex.name.replace(/\.[^.]+$/, ''), text: ex.note });
    });
    $('sample-thumbs').appendChild(b);
  }
}).catch(() => {});
</script>
</body>
</html>


In [ ]:
%%writefile /content/retake_coach/static/batch.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Retake Coach: many photos</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible+Next:wght@400;600;800&display=swap" rel="stylesheet">
<style>
:root {
  --room: #171c22; --panel: #202830; --line: #34414d; --ink: #e6ebf0; --ink-2: #9eacba;
  --keep: #63c29e; --retake: #f4b740; --away: #aab5c2; --focus: #8fc1ff;
  color-scheme: dark;
}
* { box-sizing: border-box; }
html, body { margin: 0; background: var(--room); color: var(--ink); }
body { font-family: "Atkinson Hyperlegible Next", "Atkinson Hyperlegible", system-ui, sans-serif; font-size: 17px; line-height: 1.5; -webkit-font-smoothing: antialiased; }
:focus-visible { outline: 3px solid var(--focus); outline-offset: 3px; border-radius: 6px; }
.sr { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
header { display: flex; align-items: center; justify-content: space-between; gap: 16px; flex-wrap: wrap; padding: 18px clamp(16px, 4vw, 40px); border-bottom: 1px solid var(--line); }
.left { display: flex; align-items: center; gap: 28px; flex-wrap: wrap; }
.mark { display: flex; align-items: center; gap: 10px; font-weight: 800; font-size: 21px; letter-spacing: -0.01em; }
.mark svg { width: 26px; height: 26px; }
nav { display: flex; gap: 6px; font-size: 15px; }
nav a { color: var(--ink-2); text-decoration: none; padding: 4px 2px; margin: 0 8px 0 0; border-bottom: 2px solid transparent; }
nav a:hover { color: var(--ink); }
nav a[aria-current] { color: var(--ink); border-bottom-color: var(--keep); }
.status { display: flex; gap: 8px; flex-wrap: wrap; font-size: 14px; color: var(--ink-2); }
.status span { border: 1px solid var(--line); border-radius: 999px; padding: 4px 12px; }
.status b { color: var(--ink); font-weight: 600; }

main { max-width: 1160px; margin: 0 auto; padding: clamp(20px, 4vw, 44px) clamp(16px, 4vw, 40px); }
h1 { font-size: 33px; line-height: 1.2; margin: 0 0 10px; font-weight: 800; letter-spacing: -0.015em; }
.lead { color: var(--ink-2); max-width: 64ch; margin: 0 0 24px; }
.drop { border: 2px dashed var(--line); border-radius: 18px; padding: 28px; display: flex; align-items: center; gap: 18px; flex-wrap: wrap; }
.drop.dragging { border-color: var(--focus); border-style: solid; }
.drop .lead2 { font-size: 21px; font-weight: 600; margin-right: auto; }
.btn { background: var(--ink); color: var(--room); font: inherit; font-weight: 800; border: 0; border-radius: 999px; padding: 10px 20px; cursor: pointer; }
.btn.ghost { background: none; color: var(--ink); border: 1px solid var(--line); font-weight: 600; }
.btn:disabled { opacity: 0.45; cursor: default; }
.options { display: flex; gap: 18px; align-items: center; flex-wrap: wrap; margin: 16px 0 26px; font-size: 15px; color: var(--ink-2); }
.options label { display: flex; gap: 8px; align-items: center; }
.options input { width: 18px; height: 18px; accent-color: var(--keep); }
.watch { border-left: 4px solid var(--keep); padding: 8px 14px; margin: 0 0 22px; background: var(--panel); border-radius: 0 10px 10px 0; }
.watch code { color: var(--ink); }
.summary { font-size: 19px; margin: 0 0 14px; }
.summary .k { color: var(--keep); font-weight: 600; } .summary .r { color: var(--retake); font-weight: 600; }
.progress { color: var(--ink-2); font-size: 15px; min-height: 1.5em; }
table.results { width: 100%; border-collapse: collapse; font-size: 15px; }
.results th { text-align: left; color: var(--ink-2); font-weight: 600; padding: 8px 12px 8px 0; border-bottom: 1px solid var(--line); }
.results td { padding: 10px 12px 10px 0; border-bottom: 1px solid var(--line); vertical-align: middle; }
.results img { width: 44px; height: 44px; border-radius: 50%; display: block; object-fit: cover; border: 2px solid var(--line); }
.results tr.keep img { border-color: var(--keep); } .results tr.retake img { border-color: var(--retake); }
.results tr.not_fundus img { border-color: var(--away); border-style: dashed; }
.chip { font-weight: 600; white-space: nowrap; }
.keep .chip { color: var(--keep); } .retake .chip { color: var(--retake); } .not_fundus .chip, .unreadable .chip { color: var(--away); }
.file { color: var(--ink-2); word-break: break-all; max-width: 18ch; }
.what { max-width: 52ch; }
.empty { color: var(--ink-2); padding: 18px 0; }
.fname { display: none; color: var(--ink-2); font-weight: 400; font-size: 13px; word-break: break-all; }
.watch code { word-break: break-all; }
@media (max-width: 760px) {
  .results .eye, .results .time, .results .file { display: none; }
  .fname { display: block; }
}
.note { margin: 16px 0 6px; background: var(--panel); border-left: 4px solid var(--keep); border-radius: 0 10px 10px 0; padding: 12px 16px; max-width: 72ch; }
.note p { margin: 0; }
.note .src { color: var(--ink-2); font-size: 14px; margin-top: 6px; }
.note details { margin-top: 6px; font-size: 14px; color: var(--ink-2); }
.note summary { cursor: pointer; }
.note pre { white-space: pre-wrap; font: inherit; margin: 6px 0 0; }
footer { max-width: 1160px; margin: 0 auto; padding: 8px clamp(16px, 4vw, 40px) 32px; color: var(--ink-2); font-size: 14px; }
</style>
</head>
<body>
<header>
  <div class="left">
    <div class="mark">
      <svg viewBox="0 0 26 26" aria-hidden="true"><circle cx="13" cy="13" r="11" fill="none" stroke="#63c29e" stroke-width="2.4"/><circle cx="13" cy="13" r="4" fill="#f4b740"/></svg>
      Retake Coach
    </div>
    <nav aria-label="Pages"><a href="/">One photo</a><a href="/batch" aria-current="page">Many photos</a></nav>
  </div>
  <div class="status" aria-live="polite">
    <span id="detector-status">Detector: <b>DINOv2 on this computer</b></span>
    <span id="wording-status">Coach's note: <b>checking…</b></span>
  </div>
</header>

<main>
  <h1>Check a batch of photos</h1>
  <p class="lead">Drop the photos from a clinic session, or a whole folder. Each one gets keep or retake and a line on what to fix, so the ones to redo are clear before the photos go to a grader.</p>

  <div class="drop" id="drop">
    <span class="lead2">Drop photos or a folder here</span>
    <label class="btn" for="files" tabindex="0">Choose photos</label>
    <label class="btn ghost" for="folder" tabindex="0">Choose a folder</label>
    <input class="sr" type="file" id="files" accept="image/*" multiple>
    <input class="sr" type="file" id="folder" webkitdirectory multiple>
  </div>
  <div class="options">
    <button class="btn ghost" id="note-btn" type="button" disabled>Coach's note</button>
    <button class="btn ghost" id="csv" type="button" disabled>Download CSV</button>
    <button class="btn ghost" id="clear" type="button" disabled>Clear list</button>
  </div>

  <p class="watch" id="watch" hidden></p>
  <p class="summary" id="summary"></p>
  <div class="note" id="note" aria-live="polite" hidden></div>
  <p class="progress" id="progress" aria-live="polite"></p>
  <table class="results">
    <thead><tr><th><span class="sr">Photo</span></th><th class="file">File</th><th>Result</th><th>What to fix</th><th class="eye">Eye</th><th class="eye">Centred on</th><th class="time">Checked</th></tr></thead>
    <tbody id="rows"><tr><td colspan="7" class="empty">Nothing checked yet.</td></tr></tbody>
  </table>
</main>

<footer>Checks photo quality only. It does not look for disease. Detection runs on this computer, and photos stay on it. The coach's note is written by Muse Spark (Meta Model API) from this session's counts in words.</footer>

<script>
const $ = id => document.getElementById(id);
const IMAGE = /\.(jpe?g|png|tiff?|bmp|webp)$/i;
const HEAD = { keep: 'Keep', retake: 'Retake', not_fundus: 'Not a fundus photo', unreadable: 'Unreadable' };
let rows = [], queue = [], busy = false, watchedSince = 0;

function esc(s) { return String(s ?? '').replace(/[&<>"]/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;' }[c])); }
function money(x) { return x === 0 ? '$0' : '$' + (x < 0.01 ? x.toFixed(4) : x.toFixed(2)); }
function cap(s) { return s ? s[0].toUpperCase() + s.slice(1) : '—'; }

function setStatus(b) {
  const el = $('wording-status');
  if (!b.muse) el.innerHTML = "Coach's note: <b>standard (no Muse Spark key)</b>";
  else if (!b.can_call) el.innerHTML = "Coach's note: <b>standard (today's Muse Spark cap reached)</b>";
  else el.innerHTML = `Coach's note: <b>Muse Spark</b>, ${money(b.spent_today)} of ${money(b.cap_today)} used today`;
}

function draw() {
  const n = rows.length, k = rows.filter(r => r.verdict === 'keep').length, re = rows.filter(r => r.verdict === 'retake').length;
  const away = n - k - re;
  $('summary').innerHTML = n ? `${n} checked: <span class="k">${k} keep</span>, <span class="r">${re} retake</span>`
    + (away ? `, ${away} not fundus photos or unreadable` : '') + '.' : '';
  $('csv').disabled = !n; $('note-btn').disabled = !n; $('clear').disabled = !n; $('note').hidden = true;
  $('rows').innerHTML = n ? rows.map(r => `<tr class="${r.verdict}">
      <td>${r.preview ? `<img alt="" src="${r.preview}">` : ''}</td>
      <td class="file">${esc(r.name)}</td>
      <td class="chip">${HEAD[r.verdict] || r.verdict}<span class="fname">${esc(r.name)}</span></td>
      <td class="what">${esc(r.instruction)}</td>
      <td class="eye">${cap(r.eye)}</td><td class="eye">${r.field ? (r.field === 'disc' ? 'Optic disc' : 'Macula') : '—'}</td>
      <td class="time">${esc(r.arrived || '')}</td></tr>`).join('')
    : '<tr><td colspan="7" class="empty">Nothing checked yet.</td></tr>';
}

async function pump() {                      // one photo at a time, in the order dropped
  if (busy) return; busy = true;
  let done = 0;
  while (queue.length) {
    const f = queue.shift(); done++;
    $('progress').textContent = `Checking ${f.name} (${queue.length} left)`;
    const fd = new FormData(); fd.append('photo', f, f.name); fd.append('keep_in', 'batch'); fd.append('name', f.webkitRelativePath || f.name);
    try {
      const res = await fetch('/api/check', { method: 'POST', body: fd });
      const body = await res.json().catch(() => ({}));
      const r = res.ok ? body : { verdict: 'unreadable', instruction: body.detail || 'The check did not finish.' };
      r.name = f.webkitRelativePath || f.name; r.arrived = new Date().toTimeString().slice(0, 8);
      rows.unshift(r); if (r.budget) setStatus(r.budget);
    } catch (e) {
      rows.unshift({ name: f.name, verdict: 'unreadable', instruction: 'The check server is not reachable. Make sure app.py is running.' });
    }
    draw();
  }
  $('progress').textContent = done ? `Done: ${done} photo${done === 1 ? '' : 's'} checked.` : '';
  busy = false;
}

function add(files) {
  const imgs = [...files].filter(f => IMAGE.test(f.name));
  if (!imgs.length) { $('progress').textContent = 'No photos found there. Use JPEG, PNG or TIFF files.'; return; }
  queue.push(...imgs); pump();
}

async function walk(entry, out) {            // a dropped folder: read every file inside it
  if (entry.isFile) return new Promise(res => entry.file(f => { out.push(f); res(); }, res));
  const reader = entry.createReader();
  for (;;) {
    const batch = await new Promise(res => reader.readEntries(res, () => res([])));
    if (!batch.length) break;
    for (const e of batch) await walk(e, out);
  }
}

$('files').addEventListener('change', e => { add(e.target.files); e.target.value = ''; });
$('folder').addEventListener('change', e => { add(e.target.files); e.target.value = ''; });
for (const l of document.querySelectorAll('label.btn')) l.addEventListener('keydown', e => { if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); l.click(); } });
const drop = $('drop');
['dragenter', 'dragover'].forEach(t => drop.addEventListener(t, e => { e.preventDefault(); drop.classList.add('dragging'); }));
['dragleave', 'drop'].forEach(t => drop.addEventListener(t, e => { e.preventDefault(); drop.classList.remove('dragging'); }));
drop.addEventListener('drop', async e => {
  const out = [], entries = [...e.dataTransfer.items].map(i => i.webkitGetAsEntry && i.webkitGetAsEntry()).filter(Boolean);
  if (entries.length) { for (const en of entries) await walk(en, out); add(out); } else add(e.dataTransfer.files);
});


// The coach's note: only each result's verdict, problems, obstruction place and tone, eye and field are sent,
// and the server passes only their counts in words to Muse Spark.
const slim = r => ({ verdict: r.verdict, problems: r.problems || [], eye: r.eye || null, field: r.field || null,
                     obstruction: r.obstruction ? { where: r.obstruction.where, tone: r.obstruction.tone } : null });
async function coachNote(results, box, btn) {
  const label = btn.textContent; btn.disabled = true; btn.textContent = 'Writing…';
  try {
    const res = await fetch('/api/note', { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ results: results.map(slim) }) });
    const b = await res.json();
    if (!res.ok) throw new Error(b.detail || 'failed');
    const src = b.source === 'muse-spark' ? `Written by Muse Spark for ${money(b.cost)}. No photos were sent.`
              : `Standard summary${b.source.startsWith('template (') ? ` (${esc(b.source.slice(10, -1))})` : ''}.`;
    box.innerHTML = `<p>${esc(b.note)}</p><p class="src">${src}</p>
      <details><summary>What Muse Spark was given</summary><pre>${esc(b.facts)}</pre></details>`;
    if (b.budget) setStatus(b.budget);
  } catch (e) {
    box.innerHTML = '<p>The note could not be written. Make sure app.py is running.</p>';
  }
  box.hidden = false; btn.textContent = label; btn.disabled = false;
}

$('note-btn').addEventListener('click', () => coachNote(rows, $('note'), $('note-btn')));

$('csv').addEventListener('click', () => {
  const cols = ['file', 'result', 'problems', 'what to fix', 'eye', 'centred on', 'score not good', 'focus', 'aim', 'obstruction', 'energy', 'wording'];
  const q = v => `"${String(v ?? '').replace(/"/g, '""')}"`;
  const lines = [cols.join(',')].concat(rows.map(r => [r.name, HEAD[r.verdict] || r.verdict, (r.problems || []).join(' '), r.instruction, r.eye, r.field,
    r.p_not_good, r.checks?.focus?.score, r.checks?.aim?.score, r.checks?.obstruction?.score, r.energy, r.source].map(q).join(',')));
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([lines.join('\n')], { type: 'text/csv' }));
  a.download = 'retake-coach-results.csv'; a.click();
});

async function poll() {                      // photos arriving in a watched folder (app.py --watch)
  try {
    const w = await (await fetch('/api/watch?since=' + watchedSince)).json();
    if (w.folder) {
      $('watch').hidden = false;
      $('watch').innerHTML = `Watching <code>${esc(w.folder)}</code>. Photos the camera sends there appear here as they arrive.`;
      if (w.results.length) { watchedSince = w.results[w.results.length - 1].seq + 1; rows.unshift(...w.results.reverse()); draw(); }
      setTimeout(poll, 2000);
    }
  } catch (e) { setTimeout(poll, 5000); }
}

$('clear').addEventListener('click', () => {
  rows = []; draw(); $('progress').textContent = '';
  fetch('/api/clear?which=batch', { method: 'POST' }).catch(() => {});
});

// Coming back from the other page, or reloading: bring back the list checked so far.
fetch('/api/state').then(r => r.json()).then(st => { if (st.batch.length) { rows = st.batch.slice().reverse().concat(rows); draw(); } })
.catch(() => {}).then(() => fetch('/api/status')).then(r => r.json()).then(s => {
  setStatus(s.budget); if (s.watching) poll();
  if (s.obstruction_map && s.obstruction_map !== s.detector) $('detector-status').innerHTML = 'Detector: <b>DINOv2 + DINOv3 on this computer</b>';
}).catch(() => {
  $('wording-status').innerHTML = "Coach's note: <b>server not reachable</b>";
});
</script>
</body>
</html>


Sample photos for the page, saved to `demo_examples/` in your Drive so this runs once.
They are DeepDRiD test photos the tool judged the same way as the dataset's own labels, plus a widefield photo,
a fundus photo from the web and an everyday picture. DeepDRiD is CC BY-SA 4.0.

In [ ]:
EX = f'{D}/demo_examples'
EXTRA = ['354_r1', '354_l1']   # disc centred, macula at one edge: measured aim directions for the video
fresh = not os.path.exists(f'{EX}/labels.json')
missing = [i for i in EXTRA if not os.path.exists(f'{EX}/{i}.jpg')]
if fresh or missing:
    import numpy as np, pandas as pd, shutil
    from PIL import Image
    sys.path.insert(0, APP); import coach_core as cc
    DD = '/content/deepdrid_demo'
    T = 'regular_fundus_images/Online-Challenge1&2-Evaluation'
    UWF = 'ultra-widefield_images/ultra-widefield-validation/Images/34/34_l2.jpg'
    def sh(c): subprocess.run(c, shell=True, check=True)
    if not os.path.exists(DD):
        sh(f'git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/deepdrdoc/DeepDRiD.git {DD}')
        sh(f'cd {DD} && git config core.sparseCheckout true')
    def checkout(paths):
        with open(f'{DD}/.git/info/sparse-checkout', 'w') as f:
            f.write(''.join('/' + p + '\n' for p in paths))
        sh(f'cd {DD} && git read-tree -mu HEAD')
    checkout([f'{T}/Challenge2_labels.xlsx'])
    te = pd.read_excel(f'{DD}/{T}/Challenge2_labels.xlsx')
    G = np.load(f'{D}/quality/build_features.npz')
    coach = cc.Coach.load(MODEL)
    V = coach.verdict(G['te_X'], G['te_C'])
    bad = (te['Overall quality'] == 0).values
    lab = {'focus': (te['Clarity'] <= 6).values, 'aim': (te['Field definition'] <= 6).values, 'obstruction': (te['Artifact'] >= 6).values}
    def note(i):
        found = [k for k in cc.PROBLEMS if lab[k][i]]
        return 'DeepDRiD label: ' + ('not good enough' if bad[i] else 'good') + (f" ({', '.join(found)} flagged)" if found else '')
    picks = []
    if fresh:
        picks = [i for i, v in enumerate(V) if not v['retake'] and not bad[i] and not any(lab[k][i] for k in lab)][:2]
        for k in cc.PROBLEMS:
            picks += [i for i, v in enumerate(V) if v['retake'] and v['problems'][:1] == [k] and lab[k][i] and bad[i]][:2]
    picks += [int(np.where(te.image_id == i)[0][0]) for i in (EXTRA if fresh else missing) if (te.image_id == i).any()]
    picks = list(dict.fromkeys(picks))
    files = [f"{T}/Images/{te.image_id[i].split('_')[0]}/{te.image_id[i]}.jpg" for i in picks]
    checkout([f'{T}/Challenge2_labels.xlsx', UWF] + files)   # (UWF is small; fetched again harmlessly)
    os.makedirs(EX, exist_ok=True)
    notes = {} if fresh else json.load(open(f'{EX}/labels.json'))
    for i, f in zip(picks, files):
        shutil.copy(f'{DD}/{f}', f'{EX}/{te.image_id[i]}.jpg'); notes[f'{te.image_id[i]}.jpg'] = note(i)
    if fresh:
        shutil.copy(f'{DD}/{UWF}', f'{EX}/widefield_34_l2.jpg'); notes['widefield_34_l2.jpg'] = 'DeepDRiD widefield photo (a different kind of camera)'
        import skimage.data as sd
        for name, fn, txt in [('web_retina.jpg', 'retina', 'Fundus photo from the web (scikit-image sample)'),
                              ('everyday_cat.jpg', 'chelsea', 'Everyday picture (scikit-image sample)')]:
            try:
                Image.fromarray(np.asarray(getattr(sd, fn)())[..., :3]).save(f'{EX}/{name}', quality=92); notes[name] = txt
            except Exception as e:
                print('skipped', name, type(e).__name__)
    json.dump(notes, open(f'{EX}/labels.json', 'w'), indent=1)
subprocess.run(f'rm -rf {APP}/examples && cp -r "{EX}" {APP}/examples', shell=True, check=True)
print(json.load(open(f'{APP}/examples/labels.json')))

Start the page. Re-running this cell restarts it with the same caps.

In [ ]:
USE_MUSE = True                                   # Muse Spark writes the coach's note; False: $0, a standard summary
DAILY_USD, DAILY_REQUESTS, TOTAL_USD = 0.25, 60, 1.00
MODEL_FILE = 'coach_model.npz'                    # 'coach_model_eyer2.npz' once eyer_check.ipynb has onboarded the Eyer2
WATCH = f'{D}/eyer_inbox'                         # photos that land here are checked as they arrive; None to switch off

import threading, importlib, uvicorn
os.chdir(APP); sys.path.insert(0, APP)
import coach_core as cc, app as appmod
importlib.reload(cc); importlib.reload(appmod)
client = None
if USE_MUSE:
    from google.colab import userdata
    from openai import OpenAI
    try:
        client = OpenAI(base_url='https://api.meta.ai/v1', api_key=userdata.get('MODEL_API_KEY'), timeout=30)
    except Exception as e:
        print("No MODEL_API_KEY in Colab Secrets, so the coach's note is a standard summary:", type(e).__name__)
if 'server' in globals():
    server.should_exit = True; thread.join(timeout=10)
if 'watcher' in globals() and watcher:
    watcher.stop()
budget = appmod.Budget(client, DAILY_USD, DAILY_REQUESTS, TOTAL_USD)
coach = cc.Coach.load(f'{D}/quality/{MODEL_FILE}')
if os.path.exists(LOCATOR):                                 # measured disc and macula positions for aim directions
    coach.attach_locator(cc.Locator.load(LOCATOR))
if 'encoder' not in globals() or encoder.name != coach.backbone:
    if 'dinov3' in coach.backbone:                          # gated on Hugging Face: needs HF_TOKEN in Colab Secrets
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    encoder = cc.Encoder(coach.backbone)
omap = cc.ObstructionMap.load(MAP) if os.path.exists(MAP) else None    # where an obstruction is, on obstruction retakes
if omap is None:
    map_encoder = None
elif omap.backbone == coach.backbone:
    map_encoder = encoder
elif globals().get('map_encoder') is None or map_encoder.name != omap.backbone:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')                # DINOv3 is gated on Hugging Face
        map_encoder = cc.Encoder(omap.backbone)
    except Exception as e:
        print('obstruction map off:', type(e).__name__, e); omap = map_encoder = None
print('encoder:', coach.backbone, '| retake rule:', coach.rule, '| aim directions:', 'measured' if coach.locator else 'standard wording',
      '| obstruction map:', omap.backbone if omap else 'off', '| cameras onboarded:', [c['name'] for c in coach.cameras] or 'none')
checker = appmod.Checker(coach, encoder, budget, omap, map_encoder)
watcher = None
if WATCH:
    os.makedirs(WATCH, exist_ok=True)
    watcher = appmod.Watcher(WATCH, checker).start()
server = uvicorn.Server(uvicorn.Config(appmod.create_app(coach, encoder, budget, 'examples', watcher, checker),
                                       host='127.0.0.1', port=8000, log_level='warning'))
thread = threading.Thread(target=server.run, daemon=True); thread.start()

import time; time.sleep(3)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000, path='/')        # one photo at a time, with the visit checklist
serve_kernel_port_as_window(8000, path='/batch')   # many photos, plus anything that lands in WATCH
print("Muse Spark coach's note is", 'on, capped at' if client else 'off', f'${DAILY_USD}/day, {DAILY_REQUESTS} requests/day, ${TOTAL_USD} total' if client else '')

Spending so far (the page's header shows it too):

In [ ]:
budget.status()